# DAR + STEP-inspired graph pilot — VILA1.5-3B

Bản copy của `dar_kaggle_stsg_pilot.ipynb`. Đơn vị train vẫn là **video gốc**.

`video → PySceneDetect → scene clips → clustering keyframe → frame graphs →
dynamic merging trong clip → cross-clip bridging → graph toàn video → auxiliary SFT`

Mỗi video có 2 mẫu: video → DAR JSON, và keyframes + timestamps → graph JSON.
Hai tác vụ dùng chung Qwen2.5-VL-3B; chưa có graph encoder/head riêng.
Inference DAR vẫn dùng video, không cần teacher hoặc graph.

**VILA1.5-3B** được chọn cho pilot vì hỗ trợ multi-image và phù hợp nhiều lần gọi
parse/đối chiếu ảnh. STEP dùng VILA3B và VideoChat2-Mistral7B, nhưng không cung cấp
so sánh trực tiếp chất lượng graph teacher. Không suy ra VILA tốt hơn từ kích thước.
Đây là adaptation cách dựng graph; chưa tái tạo QRA/CoT self-training của STEP.
Các prompt trong notebook do dự án viết, không phải prompt nguyên văn từ paper.

Mặc định `smoke`: 8 train / 4 dev. Chưa chạy GPU với checkpoint thật cho bản này.
Teacher chỉ nhận ID/path/duration train. Scene cut không được coi là ranh giới cảm xúc.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

# Sửa các đường dẫn theo Kaggle Inputs. BASE_MODEL phải là backbone Instruct gốc.
TEACHER_MODEL = Path('/kaggle/input/vila15-3b/VILA1.5-3b')
TEACHER_BUNDLE = Path('/kaggle/input/dar-step-vila-runtime')
TEACHER_GPUS = '0'  # Native SigLIP teacher uses one GPU, FP16.
SCENE_THRESHOLD = 27.0
MIN_SCENE_SECONDS = 0.5
# Hecate: all frames; global K=min(N//2, scene count); one frame/subshot.
MAX_KEYFRAMES = 24  # Video-dependent count; exceeding this is a recorded rejection.
MAX_LENGTH = 16384  # Preflight encodes ALL SFT rows before training.
BASE_MODEL = Path('/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2')
TRAIN_JSONL = Path('/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl')
TEST_JSONL = Path('/kaggle/input/datasets/vucongaaa/dar-annotation/test.jsonl')
VIDEO_ROOT = Path('/kaggle/input/datasets/vucongaaa/vce-original-videos/videos')
WHEELHOUSE = Path('/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128')
RUNTIME = Path('/tmp/dar_step_student_python')
PROJECT = Path('/kaggle/working/dar_step_hecate_project')
MODE = 'smoke'  # First validate smoke; then pilot/full.
assert MODE in ('smoke', 'pilot', 'full')
TRAIN_SIZE, DEV_SIZE = ((8, 4) if MODE == 'smoke' else
                        (256, 64) if MODE == 'pilot' else (-1, 64))
NUM_TRAIN_EPOCHS = 0.5
ARM = 'stsg'  # Or stsg_no_links: same observations, temporal/reference edges removed.
assert ARM in ('stsg', 'stsg_no_links')
ARMS = [ARM]
WORK = Path('/kaggle/working/dar_step_hecate_vila_epoch05_' + MODE + '_' + ARM)
MODEL_OUTPUT_ROOT = Path('/kaggle/temp/dar_step_hecate_models_' + MODE + '_' + ARM)
LEARNING_RATE = 1e-5
GRADIENT_ACCUMULATION_STEPS = 32  # Matches dar_kaggle_sft_clean.ipynb.
SEED = 1234
for path in (TEACHER_MODEL, TEACHER_BUNDLE, BASE_MODEL, TRAIN_JSONL, TEST_JSONL, VIDEO_ROOT, WHEELHOUSE):
    assert path.exists(), f'Sửa đường dẫn chưa tồn tại: {path}'
assert (TEACHER_MODEL / 'config.json').exists()
for component in ('llm', 'vision_tower', 'mm_projector'):
    assert (TEACHER_MODEL / component / 'config.json').exists(), f'Missing VILA component: {component}'
    assert list((TEACHER_MODEL / component).glob('*.safetensors')), f'Missing weights: {component}'
assert (TEACHER_BUNDLE / 'manifest.json').exists(), 'Run prepare_runtime.py on an online machine first' 
assert (BASE_MODEL / 'config.json').exists()
assert list(BASE_MODEL.glob('*.safetensors')), 'Cần full base weights, không phải LoRA adapter'

# The Kaggle model mount may omit preprocessor_config.json. Keep the read-only
# mount intact and make a lightweight local view of its files for ms-swift.
BASE_MODEL_SOURCE = BASE_MODEL
BASE_MODEL = Path('/tmp/dar_step_base_model')
BASE_MODEL.mkdir(parents=True, exist_ok=True)
for source in BASE_MODEL_SOURCE.iterdir():
    if source.name != 'preprocessor_config.json':
        target = BASE_MODEL / source.name
        if not target.exists():
            target.symlink_to(source, target_is_directory=source.is_dir())
preprocessor_source = BASE_MODEL_SOURCE / 'preprocessor_config.json'
preprocessor = (json.loads(preprocessor_source.read_text()) if preprocessor_source.exists()
                else dict(min_pixels=3136, max_pixels=12845056, patch_size=14,
                          temporal_patch_size=2, merge_size=2,
                          image_mean=[0.48145466, 0.4578275, 0.40821073],
                          image_std=[0.26862954, 0.26130258, 0.27577711]))
preprocessor['image_processor_type'] = 'Qwen2VLImageProcessor'
preprocessor.setdefault('processor_class', 'Qwen2_5_VLProcessor')
(BASE_MODEL / 'preprocessor_config.json').write_text(json.dumps(preprocessor))
print('Student model view:', BASE_MODEL, 'preprocessor:', preprocessor)


Runtime riêng, dùng wheelhouse có sẵn; không thay NumPy/Torch đang load trong
kernel. Tất cả tác vụ ML chạy ở subprocess. Không thêm `ms-swift/` checkout 4.0
vào PYTHONPATH của runtime 3.12.5.

In [ ]:
"""Student bootstrap cell; embedded verbatim by build_notebook.py."""
import tempfile
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

# Preserve Kaggle's package directories (including sitecustomize dependencies).
# Do not inherit an old student/teacher overlay from a previous bootstrap.
base_paths = [p for p in sys.path if p and Path(p).is_dir()
              and ('site-packages' in p or 'dist-packages' in p)]
inherited_paths = os.environ.get('PYTHONPATH', '').split(os.pathsep)
base_paths = list(dict.fromkeys(
    p for p in [*inherited_paths, *base_paths] if p
    and not Path(p).resolve().is_relative_to(Path('/kaggle/working'))))
base_env = dict(os.environ, PYTHONPATH=os.pathsep.join(base_paths),
                PIP_NO_INDEX='1', PIP_DISABLE_PIP_VERSION_CHECK='1',
                HF_HUB_OFFLINE='1', TOKENIZERS_PARALLELISM='false')

# Check the actual base interpreter, not torch already imported in the kernel.
subprocess.run([sys.executable, '-c',
    "import torch; print('Base torch:', torch.__version__, torch.__file__); "
    "assert torch.version.cuda is not None, 'Kaggle base Torch is CPU-only; use a GPU image with CUDA Torch'; "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing'"],
    check=True, env={**base_env, 'CUDA_VISIBLE_DEVICES': TEACHER_GPUS})

supported_tags = set(sys_tags())
wheels = []
student_pins = {'transformers': '4.57.1', 'ms-swift': '3.12.5'}
for wheel in sorted(WHEELHOUSE.rglob('*.whl')):
    name, version, _, tags = parse_wheel_filename(wheel.name)
    # Reuse Kaggle's matched CUDA stack; never overlay it with a CPU wheel.
    if name in {'torch', 'torchvision', 'torchaudio', 'triton', 'torchao'} or name.startswith('nvidia-'):
        continue
    if not (tags & supported_tags) or (name == 'packaging' and str(version) == '26.3'):
        continue
    # Do not pass two versions of one package to pip if additional wheels are attached.
    if name in student_pins and str(version) != student_pins[name]:
        continue
    wheels.append(wheel)
assert wheels, f'No compatible student wheels in {WHEELHOUSE}'
assert any(parse_wheel_filename(p.name)[0] == 'transformers'
           and str(parse_wheel_filename(p.name)[1]) == student_pins['transformers']
           for p in wheels), 'Student wheelhouse needs transformers 4.57.1'

# A fresh overlay prevents CPU Torch left by an earlier cell from being imported.
RUNTIME.mkdir(parents=True, exist_ok=True)
student_site = Path(tempfile.mkdtemp(prefix='student-', dir=RUNTIME))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--only-binary=:all:', '--target', str(student_site), *map(str, wheels)],
               check=True, env=base_env)
env = dict(base_env, PYTHONPATH=os.pathsep.join([str(student_site), *base_paths]),
           CUDA_VISIBLE_DEVICES='0')
def run(*args, extra_env=None):
    subprocess.run(list(map(str, args)), check=True, env={**env, **(extra_env or {})})
run(sys.executable, '-c',
    "import torch, transformers, swift, peft, qwen_vl_utils; "
    "print('Student:', torch.__version__, transformers.__version__, swift.__version__); "
    "print('Torch path:', torch.__file__); print('Transformers path:', transformers.__file__); "
    "assert torch.version.cuda is not None and torch.cuda.is_available(), 'Student requires CUDA Torch'; "
    "assert transformers.__version__ == '4.57.1', transformers.__file__; "
    "assert swift.__version__ == '3.12.5'; print(torch.cuda.get_device_name(0))")


In [ ]:
run(sys.executable, '-c',
    "import sys; from transformers import AutoProcessor; "
    "p = AutoProcessor.from_pretrained(sys.argv[1], local_files_only=True); "
    "print('Student processor preflight:', type(p).__name__, type(p.image_processor).__name__)",
    BASE_MODEL)


## Runtime teacher riêng và dữ liệu cần attach

Giữ student runtime của pilot: Transformers 4.57.1 / ms-swift 3.12.5.
Teacher dùng native VILA revision `6b941da19e31ddfdfaa60160908ccf0978d96615`,
Transformers 4.36.2, FP16, eager attention. Không cài patch training FlashAttention.
Imports của các vision tower không dùng được chuyển sang lazy import; nguồn và diff
được lưu trong `vila/dar-runtime.json`. CUDA Torch của Kaggle được giữ nguyên.
Stack Torch mới của Kaggle cần được kiểm tra bằng smoke, chưa có kiểm chứng GPU thật.

Chuẩn bị trên máy có Internet:
```
python experiments/dar_step/prepare_runtime.py --output experiments/dar_step/runtime-bundle
```
Attach thư mục output thành dataset `dar-step-vila-runtime`. Script chỉ tải code/wheels,
không tải weights. Attach riêng snapshot đầy đủ
[`Efficient-Large-Model/VILA1.5-3b`](https://huggingface.co/Efficient-Large-Model/VILA1.5-3b)
với `llm/`, `vision_tower/`, `mm_projector/` và tokenizer trong `llm/`.
Sửa 2 đường dẫn teacher trong config. Notebook chạy Internet Off, bundle Python 3.12/Linux.

Không dùng runtime VideoLLaMA3 cũ cho VILA. Cell dưới kiểm tra hash bundle và native imports.

In [ ]:
"""Offline teacher bootstrap, embedded in the notebook; never import ML in kernel."""
import hashlib
import tempfile
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

bundle_meta = json.loads((TEACHER_BUNDLE / 'manifest.json').read_text())
assert bundle_meta['revision'] == '6b941da19e31ddfdfaa60160908ccf0978d96615'
for name, expected in bundle_meta['files'].items():
    path = (TEACHER_BUNDLE / name).resolve()
    assert path.is_relative_to(TEACHER_BUNDLE.resolve()), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'Bundle changed: {path}'
teacher_site = Path(tempfile.mkdtemp(prefix='dar-vila-site-'))
tags = set(sys_tags())
teacher_wheels = []
for path in sorted((TEACHER_BUNDLE / 'wheels').glob('*.whl')):
    name, _, _, wheel_tags = parse_wheel_filename(path.name)
    assert name not in ('torch', 'torchvision', 'torchaudio', 'triton') and not name.startswith('nvidia-')
    if tags & wheel_tags:
        teacher_wheels.append(path)
assert teacher_wheels, 'Prepare the bundle for Kaggle Python 3.12 / Linux'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--target', str(teacher_site), *map(str, teacher_wheels)], check=True, env=base_env)
teacher_python = sys.executable
teacher_env = dict(base_env, PYTHONPATH=os.pathsep.join(
    [str(TEACHER_BUNDLE / 'vila'), str(teacher_site), *base_paths]), CUDA_VISIBLE_DEVICES=TEACHER_GPUS)
subprocess.run([teacher_python, '-c',
    "import torch, transformers, scenedetect, cv2; from llava.model.builder import load_pretrained_model; "
    "assert transformers.__version__ == '4.36.2'; assert scenedetect.__version__ == '0.6.5.2'; "
    "assert torch.cuda.is_available(); print('VILA native imports OK:', transformers.__file__, torch.__version__)"],
    check=True, env=teacher_env)


Cell sau giải nén code đã nhúng vào notebook; không phụ thuộc bản DAR cũ trong
Kaggle Inputs. Các file hiện có khác nội dung sẽ làm cell dừng để tránh trộn version.

In [ ]:
FILES = {'experiments/dar_stsg/core.py': '"""Training-only STSG auxiliary supervision. No torch dependency in data tools."""\nfrom __future__ import annotations\n\nimport ast\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[2]\nARMS = (\'baseline\', \'caption\', \'stsg\', \'stsg_no_links\')\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\n\ndef write_jsonl(path, rows):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as stream:\n        for row in rows:\n            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef official_functions():\n    """Load pure prompt/scoring functions without importing the vLLM entry point."""\n    names = {\'EMOTION_CANDIDATES\', \'build_eval_prompt\', \'calculate_iou\',\n             \'evaluate_single_video\', \'compute_overall_metrics\',\n             \'merge_adjacent_same_emotion_segments\', \'validate_and_fix_segments\'}\n    tree = ast.parse((ROOT / \'test.py\').read_text(encoding=\'utf-8\'))\n    selected = []\n    for node in tree.body:\n        if isinstance(node, ast.FunctionDef) and node.name in names:\n            selected.append(node)\n        elif isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id in names for t in node.targets):\n            selected.append(node)\n    from typing import List, Dict\n    scope = {\'List\': List, \'Dict\': Dict}\n    exec(compile(ast.Module(body=selected, type_ignores=[]), str(ROOT / \'test.py\'), \'exec\'), scope)\n    return scope\n\n\ndef number(x):\n    if isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x):\n        raise ValueError(\'Expected a finite numeric value\')\n    return float(x)\n\n\ndef parse_json(text):\n    text = text.strip()\n    if text.startswith(\'```\') and text.endswith(\'```\'):\n        text = text.split(\'\\n\', 1)[1].rsplit(\'```\', 1)[0].strip()\n    obj = json.loads(text)\n    if not isinstance(obj, dict):\n        raise ValueError(\'Expected a JSON object\')\n    return obj\n\n\ndef validate_graph(graph, duration):\n    """Structural validation only; it does not certify visual truth."""\n    if set(graph) != {\'entities\', \'events\', \'temporal_links\'}:\n        raise ValueError(\'Invalid STSG top-level keys\')\n    entities, events, links = (graph[k] for k in (\'entities\', \'events\', \'temporal_links\'))\n    if not all(isinstance(x, list) for x in (entities, events, links)):\n        raise ValueError(\'Graph fields must be lists\')\n    if not 1 <= len(entities) <= 8 or not 1 <= len(events) <= 8 or len(links) > 16:\n        raise ValueError(\'Graph size outside pilot limits\')\n    entity_ids = set()\n    for entity in entities:\n        if set(entity) != {\'id\', \'label\'} or not all(isinstance(v, str) and v.strip() for v in entity.values()):\n            raise ValueError(\'Invalid entity\')\n        if entity[\'id\'] in entity_ids:\n            raise ValueError(\'Duplicate entity ID\')\n        entity_ids.add(entity[\'id\'])\n    event_ids = {}\n    for event in events:\n        if set(event) != {\'id\', \'start_time\', \'end_time\', \'observation\', \'relations\'}:\n            raise ValueError(\'Invalid event keys\')\n        if not isinstance(event[\'id\'], str) or not event[\'id\'] or event[\'id\'] in event_ids:\n            raise ValueError(\'Invalid/duplicate event ID\')\n        start, end = number(event[\'start_time\']), number(event[\'end_time\'])\n        if not 0 <= start < end <= duration + .051:\n            raise ValueError(\'Event outside video\')\n        if not isinstance(event[\'observation\'], str) or not event[\'observation\'].strip():\n            raise ValueError(\'Missing visual observation\')\n        if not isinstance(event[\'relations\'], list) or len(event[\'relations\']) > 8:\n            raise ValueError(\'Invalid relations\')\n        for edge in event[\'relations\']:\n            if not isinstance(edge, list) or len(edge) != 3 or not all(isinstance(v, str) for v in edge):\n                raise ValueError(\'Expected subject-predicate-object triple\')\n            if edge[0] not in entity_ids or edge[2] not in entity_ids or not edge[1].strip():\n                raise ValueError(\'Dangling relation\')\n        event_ids[event[\'id\']] = event\n    for edge in links:\n        if not isinstance(edge, list) or len(edge) != 3:\n            raise ValueError(\'Invalid temporal edge\')\n        a, kind, b = edge\n        if a not in event_ids or b not in event_ids or a == b or kind != \'before\':\n            raise ValueError(\'Invalid temporal references\')\n        if event_ids[a][\'end_time\'] > event_ids[b][\'start_time\'] + .051:\n            raise ValueError(\'Contradictory before edge\')\n    return graph\n\n\ndef evidence_prompt(duration, kind):\n    common = (f\'Observe this SILENT video of duration {duration:.1f}s. \'\n              \'Describe only visible entities, interactions, appearance, camera/scene changes and events. \'\n              \'Do not use audio, viewer emotion labels, inferred intentions or imagined outcomes. \'\n              \'Omit uncertain details. An event boundary is not necessarily an emotion boundary. \')\n    if kind == \'caption\':\n        return common + \'Return ONLY JSON {"caption":"a concise chronological visual description, at most 250 words"}.\'\n    return common + \'\'\'Return ONLY one compact JSON object, with no markdown or explanation.\nIt must have exactly three top-level keys: entities, events, temporal_links.\nentities is a nonempty array of at most 8 objects, each with exactly id and label.\nUse a distinct short string ID for each visible entity and a specific visual label.\nevents is a nonempty chronological array of at most 8 objects, each with exactly\nid, start_time, end_time, observation, relations. Use distinct event IDs.\nTimes are numeric seconds within the video, with start_time strictly before end_time.\nThe observation must describe a concrete visible state or change.\nrelations is an array of at most 8 triples of strings: subject entity ID,\nvisible relation, object entity ID. Both IDs MUST occur in entities.\ntemporal_links is an array of at most 16 triples of strings: earlier event ID,\nthe word before, later event ID. Both IDs MUST occur in events and the earlier\nevent must end no later than the later event starts.\nUse empty arrays for relations or temporal_links when no justified edge exists.\nNever invent an ID merely to fill a relation or temporal link. Never copy schema\ndescriptions into labels or observations. Do not invent bounding boxes.\'\'\'\n\n\ndef dar_prompt(duration):\n    return official_functions()[\'build_eval_prompt\'](duration)\n\n\ndef annotation(row, video_root=None):\n    video = row.get(\'video\') or row.get(\'video_path\')\n    if not isinstance(video, str):\n        raise ValueError(\'Missing video\')\n    basename = video.replace(\'\\\\\', \'/\').split(\'/\')[-1]\n    path = str(Path(video_root) / basename) if video_root else video\n    duration = number(row.get(\'video_duration\'))  # Never infer duration from GT boundaries.\n    if duration <= 0:\n        raise ValueError(\'Invalid duration\')\n    answers = [x[\'value\'] for x in row.get(\'conversations\', []) if x.get(\'from\') == \'gpt\']\n    target = parse_json(answers[-1]) if answers else {\'segments\': row.get(\'gt_segments\')}\n    if not isinstance(target.get(\'segments\'), list) or not target[\'segments\']:\n        raise ValueError(\'Missing DAR target\')\n    return dict(video_id=Path(basename).stem, video_path=path, video_duration=duration, target=target)\n\n\ndef sft_rows(row, evidence, arm):\n    def sample(prompt, target, suffix):\n        return {\'id\': row[\'video_id\'] + \':\' + suffix,\n                \'messages\': [{\'role\': \'user\', \'content\': \'<video>\\n\' + prompt},\n                             {\'role\': \'assistant\', \'content\': json.dumps(target, ensure_ascii=False)}],\n                \'videos\': [row[\'video_path\']]}\n    answer = sample(dar_prompt(row[\'video_duration\']), row[\'target\'], \'dar\')\n    if arm == \'baseline\':\n        auxiliary = copy.deepcopy(answer)\n        auxiliary[\'id\'] = row[\'video_id\'] + \':dar-repeat\'\n    elif arm == \'caption\':\n        auxiliary = sample(evidence_prompt(row[\'video_duration\'], \'caption\'), evidence[\'caption\'], \'caption\')\n    else:\n        graph = copy.deepcopy(evidence[\'stsg\'])\n        prompt = evidence_prompt(row[\'video_duration\'], \'stsg\')\n        if arm == \'stsg_no_links\':\n            graph[\'temporal_links\'] = []\n            prompt += \'\\nFor this task leave temporal_links empty; retain event timestamps.\'\n        auxiliary = sample(prompt, graph, arm)\n    return [answer, auxiliary]\n', 'experiments/dar_stsg/prepare.py': '"""Freeze pilot split, then create matched SFT arms from train-only evidence."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom core import ARMS, annotation, digest, read_jsonl, sft_rows, validate_graph, write_jsonl\n\n\ndef split(args):\n    rows = [annotation(r, args.video_root) for r in read_jsonl(args.train)]\n    ids = [r[\'video_id\'] for r in rows]\n    if len(ids) != len(set(ids)):\n        raise ValueError(\'Duplicate source video IDs; split at video level first\')\n    heldout = {annotation(r)[\'video_id\'] for r in read_jsonl(args.test)}\n    if set(ids) & heldout:\n        raise ValueError(\'Train/test video overlap\')\n    rows.sort(key=lambda r: digest([args.seed, r[\'video_id\']]))\n    train_size = len(rows) - args.dev_size if args.train_size == -1 else args.train_size\n    if train_size < 1 or args.dev_size < 1 or len(rows) < train_size + args.dev_size:\n        raise ValueError(\'Insufficient rows or invalid requested sizes\')\n    dev, train = rows[:args.dev_size], rows[args.dev_size:args.dev_size + train_size]\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    write_jsonl(out / \'train.jsonl\', train)\n    write_jsonl(out / \'dev.jsonl\', dev)\n    # The teacher receives no labels, GT event boundaries or previous model predictions.\n    write_jsonl(out / \'teacher_inputs.jsonl\', [{k: r[k] for k in (\'video_id\', \'video_path\', \'video_duration\')} for r in train])\n    (out / \'split.json\').write_text(json.dumps(dict(seed=args.seed, train=len(train), dev=len(dev),\n        train_source_sha256=digest(read_jsonl(args.train)), test_source_sha256=digest(read_jsonl(args.test)),\n        train_ids=[r[\'video_id\'] for r in train], dev_ids=[r[\'video_id\'] for r in dev]), indent=2), encoding=\'utf-8\')\n\n\ndef build(args):\n    arms = getattr(args, \'arms\', ARMS)\n    need_graph = any(a in (\'stsg\', \'stsg_no_links\') for a in arms)\n    need_caption = \'caption\' in arms\n    rows = read_jsonl(Path(args.split) / \'train.jsonl\')\n    evidence = read_jsonl(args.evidence)\n    by_id = {r[\'video_id\']: r for r in evidence}\n    if len(by_id) != len(evidence):\n        raise ValueError(\'Duplicate evidence; do not select the best retry\')\n    if set(by_id) != {r[\'video_id\'] for r in rows}:\n        raise ValueError(\'Evidence must cover exactly the frozen train IDs, including failures\')\n    accepted, rejected = [], []\n    for row in rows:\n        e = by_id[row[\'video_id\']]\n        try:\n            identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            if e.get(\'input_sha256\') != digest(identity):\n                raise ValueError(\'Evidence input identity mismatch\')\n            if need_graph:\n                validate_graph(e[\'stsg\'], row[\'video_duration\'])\n            if need_caption and (set(e[\'caption\']) != {\'caption\'} or not isinstance(e[\'caption\'][\'caption\'], str) or not e[\'caption\'][\'caption\'].strip()):\n                raise ValueError(\'Invalid caption\')\n            accepted.append(row)\n        except (KeyError, ValueError, TypeError) as exc:\n            rejected.append({\'video_id\': row[\'video_id\'], \'error\': str(exc)})\n    if not accepted:\n        raise ValueError(\'No common valid evidence; inspect teacher outputs\')\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    for arm in arms:\n        write_jsonl(out / (arm + \'.jsonl\'), [s for r in accepted for s in sft_rows(r, by_id[r[\'video_id\']], arm)])\n    (out / \'build.json\').write_text(json.dumps(dict(accepted=len(accepted), rejected=rejected,\n        arms=list(arms), source_count=len(rows), evidence_sha256=digest(evidence), ids=[r[\'video_id\'] for r in accepted],\n        note=\'Same accepted video IDs and two rows per video in every arm. Schema validation is not factual verification.\'), indent=2), encoding=\'utf-8\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    p = sub.add_parser(\'split\')\n    for flag in (\'train\', \'test\', \'video-root\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.add_argument(\'--train-size\', type=int, default=256)\n    p.add_argument(\'--dev-size\', type=int, default=64)\n    p.add_argument(\'--seed\', type=int, default=20260923)\n    p.set_defaults(run=split)\n    p = sub.add_parser(\'build\')\n    p.add_argument(\'--arms\', nargs=\'+\', choices=ARMS, default=list(ARMS))\n    for flag in (\'split\', \'evidence\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.set_defaults(run=build)\n    args = parser.parse_args()\n    args.run(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/run.py': '"""Generate train-only pseudo graphs/captions, or evaluate a fresh SFT adapter."""\nimport argparse\nimport importlib.metadata\nimport json\nimport time\nfrom pathlib import Path\n\nfrom core import ROOT, dar_prompt, digest, evidence_prompt, parse_json, read_jsonl, validate_graph\n\n\nclass Backend:\n    def __init__(self, model, adapter=None, frames=16, pixels=100352):\n        import torch\n        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\n        from qwen_vl_utils import process_vision_info\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'This pilot needs a CUDA GPU; use the Kaggle notebook\')\n        self.torch, self.vision = torch, process_vision_info\n        self.processor = AutoProcessor.from_pretrained(model)\n        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n            model, torch_dtype=torch.float16, device_map=\'auto\', attn_implementation=\'sdpa\')\n        if adapter:\n            from peft import PeftModel\n            self.model = PeftModel.from_pretrained(self.model, adapter)\n        self.model.eval()\n        self.frames, self.pixels = frames, pixels\n\n    def generate(self, row, prompt, max_tokens):\n        if not Path(row[\'video_path\']).is_file():\n            raise FileNotFoundError(row[\'video_path\'])\n        messages = [{\'role\': \'user\', \'content\': [\n            {\'type\': \'video\', \'video\': row[\'video_path\'], \'nframes\': self.frames,\n             \'min_pixels\': self.pixels, \'max_pixels\': self.pixels},\n            {\'type\': \'text\', \'text\': prompt}]}]\n        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        # Qwen2.5 needs sampled fps, not source fps. The local utility supplies it\n        # when return_video_metadata=False. Do not drop these video_kwargs.\n        # False is the utility\'s default; omitting the keyword also supports\n        # pre-Qwen3 versions which already return the required sampled fps.\n        images, videos, kwargs = self.vision(messages, return_video_kwargs=True)\n        if not kwargs.get(\'fps\'):\n            raise RuntimeError(\'Qwen2.5 video input is missing sampled fps\')\n        frame_hash = digest([self.frames, self.pixels])\n        if videos:\n            import hashlib\n            frame_hash = hashlib.sha256(videos[0].numpy().tobytes()).hexdigest()\n        inputs = self.processor(text=[text], images=images, videos=videos, padding=True,\n                                return_tensors=\'pt\', **kwargs).to(self.model.device)\n        self.torch.cuda.synchronize()\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)\n        self.torch.cuda.synchronize()\n        tokens = output[:, inputs.input_ids.shape[1]:]\n        result = dict(raw=self.processor.batch_decode(tokens, skip_special_tokens=True)[0],\n                      generated_tokens=tokens.shape[1], input_tokens=inputs.input_ids.shape[1],\n                      generation_seconds=time.monotonic() - started, frame_sha256=frame_hash,\n                      sampled_fps=kwargs.get(\'fps\'), prompt_sha256=digest(prompt),\n                      hit_token_limit=tokens.shape[1] >= max_tokens)\n        del inputs, output, tokens\n        return result\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'mode\', choices=[\'teacher\', \'predict\'])\n    for flag in (\'model\', \'input\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.add_argument(\'--adapter\')\n    p.add_argument(\'--backend\', choices=[\'qwen\', \'videollama3\'], default=\'qwen\')\n    p.add_argument(\'--kinds\', nargs=\'+\', choices=[\'caption\', \'stsg\'], default=[\'caption\', \'stsg\'])\n    p.add_argument(\'--load-in-4bit\', action=\'store_true\')\n    p.add_argument(\'--frames\', type=int, default=16)\n    p.add_argument(\'--max-tokens\', type=int, default=2048)\n    p.add_argument(\'--seed\', type=int, default=1234)\n    args = p.parse_args()\n    out = Path(args.output)\n    if out.exists() or out.with_suffix(\'.meta.json\').exists():\n        raise FileExistsError(\'Choose a new output; no silent retries or overwrites\')\n    rows = read_jsonl(args.input)\n    if not rows or len({r[\'video_id\'] for r in rows}) != len(rows):\n        raise ValueError(\'Empty manifest or duplicate video IDs\')\n    if args.mode == \'teacher\' and any(set(r) != {\'video_id\', \'video_path\', \'video_duration\'} for r in rows):\n        raise ValueError(\'Teacher input must be label-free; use teacher_inputs.jsonl\')\n    from transformers import set_seed\n    set_seed(args.seed)\n    if args.backend == \'videollama3\':\n        if args.mode != \'teacher\' or args.adapter:\n            raise ValueError(\'VideoLLaMA3 is a frozen teacher only\')\n        from teacher_videollama3 import VideoLLaMA3Backend\n        backend = VideoLLaMA3Backend(args.model, args.frames, args.load_in_4bit)\n    else:\n        if args.load_in_4bit:\n            raise ValueError(\'4-bit is currently implemented for VideoLLaMA3 teacher only\')\n        backend = Backend(args.model, args.adapter, args.frames)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    versions = {x: importlib.metadata.version(x) for x in (\'torch\', \'transformers\', \'accelerate\')}\n    meta = dict(config=vars(args), input_sha256=digest(rows), packages=versions,\n                code_sha256={p.name: digest(p.read_text(encoding=\'utf-8\')) for p in [Path(__file__), Path(__file__).with_name(\'core.py\'), ROOT / \'test.py\']})\n    if args.backend == \'videollama3\':\n        meta[\'teacher\'] = backend.metadata\n        backend_path = Path(__file__).with_name(\'teacher_videollama3.py\')\n        meta[\'code_sha256\'][backend_path.name] = digest(backend_path.read_text(encoding=\'utf-8\'))\n    out.with_suffix(\'.meta.json\').write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\n    with out.open(\'x\', encoding=\'utf-8\') as stream:\n        for i, row in enumerate(rows):\n            identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            result = dict(video_id=row[\'video_id\'], input_sha256=digest(identity), calls={})\n            for kind in (args.kinds if args.mode == \'teacher\' else (\'dar\',)):\n                prompt = dar_prompt(row[\'video_duration\']) if kind == \'dar\' else evidence_prompt(row[\'video_duration\'], kind)\n                try:\n                    call = backend.generate(identity, prompt, args.max_tokens)\n                    result[\'calls\'][kind] = call\n                    obj = parse_json(call[\'raw\'])\n                    if kind == \'stsg\':\n                        validate_graph(obj, row[\'video_duration\'])\n                    result[kind] = obj\n                except (ValueError, KeyError, TypeError) as exc:\n                    result[kind + \'_error\'] = str(exc)\n                # OOM, corrupt/missing video and model errors abort: do not silently\n                # turn a broken runtime into an apparent poor benchmark result.\n            stream.write(json.dumps(result, ensure_ascii=False, allow_nan=False) + \'\\n\')\n            stream.flush()\n            print(f\'{i + 1}/{len(rows)} {row["video_id"]}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/evaluate.py': '"""Paired, video-level evaluation. Missing/invalid outputs count as failures."""\nimport argparse\nimport copy\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nfrom core import digest, number, official_functions, read_jsonl\n\n\nOFFICIAL = official_functions()\n\n\ndef valid_segments(obj, duration):\n    if not isinstance(obj, dict) or set(obj) != {\'segments\'}:\n        return []\n    segs = obj[\'segments\']\n    if not isinstance(segs, list) or not segs:\n        return []\n    end, emotion = 0., None\n    try:\n        for s in segs:\n            start, stop = number(s[\'start_time\']), number(s[\'end_time\'])\n            if abs(start - end) > .051 or stop <= start or stop > duration + .051:\n                return []\n            if any(abs(x - round(x, 1)) > 1e-6 for x in (start, stop)):\n                return []\n            if s[\'emotion\'] not in OFFICIAL[\'EMOTION_CANDIDATES\'] or s[\'emotion\'] == emotion:\n                return []\n            if not isinstance(s[\'reason\'], str) or not s[\'reason\'].strip():\n                return []\n            end, emotion = stop, s[\'emotion\']\n        return segs if abs(end - duration) <= .051 else []\n    except (ValueError, TypeError, KeyError):\n        return []\n\n\ndef stats(pred, gt):\n    matched = 0\n    if pred and gt:\n        eligible = np.array([[OFFICIAL[\'calculate_iou\'](a, b) >= .5 and a[\'emotion\'] == b[\'emotion\'] for b in gt] for a in pred])\n        i, j = linear_sum_assignment(eligible.astype(int), maximize=True)\n        matched = int(eligible[i, j].sum())\n    return [matched, len(pred), len(gt), int(bool(pred)), int(len(pred) == len(gt))]\n\n\ndef aggregate(array):\n    tp, p, g, valid, count = array.sum(axis=0)\n    return dict(joint_f1_at_05=float(2 * tp / (p + g)) if p + g else 0.,\n                coverage=float(valid / len(array)), count_accuracy=float(count / len(array)),\n                predicted_segments=int(p), gt_segments=int(g))\n\n\ndef evaluate(manifest, predictions):\n    by_id = {r[\'video_id\']: r for r in predictions}\n    if len(by_id) != len(predictions) or set(by_id) - {r[\'video_id\'] for r in manifest}:\n        raise ValueError(\'Duplicate or out-of-manifest predictions\')\n    arrays, official_results = [], {}\n    for row in manifest:\n        prediction = by_id.get(row[\'video_id\'], {})\n        identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n        if prediction.get(\'input_sha256\') is not None and prediction[\'input_sha256\'] != digest(identity):\n            raise ValueError(\'Prediction is from a different video input manifest\')\n        obj = prediction.get(\'dar\')\n        pred = valid_segments(obj, row[\'video_duration\'])\n        gt = row[\'target\'][\'segments\']\n        arrays.append(stats(pred, gt))\n        # A separate, explicitly labelled compatibility view uses repo repairs.\n        try:\n            repaired = OFFICIAL[\'validate_and_fix_segments\'](copy.deepcopy(obj[\'segments\']), row[\'video_duration\']) if obj else []\n            result = OFFICIAL[\'evaluate_single_video\'](repaired, gt)\n            if not all(np.isfinite(v) for v in result[\'ious\']):\n                raise ValueError(\'Non-finite repaired times\')\n        except (ValueError, TypeError, KeyError, AttributeError):\n            result = OFFICIAL[\'evaluate_single_video\']([], gt)\n        official_results[row[\'video_id\']] = result\n    array = np.asarray(arrays)\n    return array, dict(strict=aggregate(array), official_repaired=OFFICIAL[\'compute_overall_metrics\'](official_results),\n                       missing_videos=len(manifest) - len(by_id))\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--manifest\', required=True)\n    p.add_argument(\'--predictions\', nargs=\'+\', required=True, help=\'name=path pairs\')\n    p.add_argument(\'--reference\', default=\'caption\')\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    args = p.parse_args()\n    manifest = read_jsonl(args.manifest)\n    if not manifest or len({r[\'video_id\'] for r in manifest}) != len(manifest):\n        raise ValueError(\'Empty or duplicate manifest\')\n    arrays, summary, frames = {}, {}, {}\n    for item in args.predictions:\n        name, path = item.split(\'=\', 1)\n        if name in arrays:\n            raise ValueError(\'Repeated arm name\')\n        predictions = read_jsonl(path)\n        for row in predictions:\n            h = row.get(\'calls\', {}).get(\'dar\', {}).get(\'frame_sha256\')\n            if h is not None and row[\'video_id\'] in frames and frames[row[\'video_id\']] != h:\n                raise ValueError(\'Mismatched sampled frames across arms\')\n            if h is not None:\n                frames[row[\'video_id\']] = h\n        arrays[name], summary[name] = evaluate(manifest, predictions)\n    if args.reference not in arrays or args.bootstrap < 1:\n        raise ValueError(\'Missing reference or invalid bootstrap count\')\n    contrasts = {}\n    for name, array in arrays.items():\n        if name == args.reference:\n            continue\n        rng = np.random.default_rng(20260923)\n        diffs = []\n        for _ in range(args.bootstrap):\n            idx = rng.integers(0, len(manifest), len(manifest))\n            diffs.append(aggregate(array[idx])[\'joint_f1_at_05\'] - aggregate(arrays[args.reference][idx])[\'joint_f1_at_05\'])\n        contrasts[name + \'_minus_\' + args.reference] = dict(\n            delta=summary[name][\'strict\'][\'joint_f1_at_05\'] - summary[args.reference][\'strict\'][\'joint_f1_at_05\'],\n            paired_video_bootstrap_95ci=np.quantile(diffs, [.025, .975]).tolist())\n    with Path(args.output).open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(dict(videos=len(manifest), arms=summary, contrasts=contrasts,\n                       note=\'Fractions, not percentages. CI conditions on training seed and selected videos; not training variance.\'), stream, indent=2)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/test_pipeline.py': 'import copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\n\nfrom core import annotation, digest, evidence_prompt, official_functions, sft_rows, validate_graph, write_jsonl, read_jsonl\nfrom evaluate import aggregate, evaluate, stats, valid_segments\nfrom prepare import build, split\n\n\ndef graph():\n    return {\'entities\': [{\'id\': \'o1\', \'label\': \'ball\'}, {\'id\': \'o2\', \'label\': \'table\'}],\n            \'events\': [{\'id\': \'e1\', \'start_time\': 0., \'end_time\': 1., \'observation\': \'Ball on table\', \'relations\': [[\'o1\', \'on\', \'o2\']]},\n                       {\'id\': \'e2\', \'start_time\': 1., \'end_time\': 2., \'observation\': \'Ball rolls\', \'relations\': []}],\n            \'temporal_links\': [[\'e1\', \'before\', \'e2\']]}\n\n\ndef segment(a=0., b=2., emotion=\'Interest\'):\n    return dict(start_time=a, end_time=b, emotion=emotion, reason=\'Visible object attracts attention.\')\n\n\ndef row(i=\'v\'):\n    return dict(video_id=i, video_path=i + \'.mp4\', video_duration=2., target={\'segments\': [segment()]})\n\n\nclass PipelineTests(unittest.TestCase):\n    def test_graph_prompt_has_no_copyable_placeholder_json(self):\n        prompt = evidence_prompt(2, \'stsg\')\n        self.assertNotIn(\'"o1"\', prompt)\n        self.assertNotIn(\'"e1"\', prompt)\n        self.assertNotIn(\'visible object or scene\', prompt)\n        self.assertIn(\'Both IDs MUST occur\', prompt)\n\n    def test_invalid_references_and_time(self):\n        self.assertEqual(validate_graph(graph(), 2), graph())\n        for change in (\'id\', \'time\', \'nan\', \'reverse\'):\n            g = graph()\n            if change == \'id\': g[\'events\'][0][\'relations\'][0][2] = \'missing\'\n            if change == \'time\': g[\'events\'][0][\'end_time\'] = 3\n            if change == \'nan\': g[\'events\'][0][\'start_time\'] = float(\'nan\')\n            if change == \'reverse\': g[\'temporal_links\'] = [[\'e2\', \'before\', \'e1\']]\n            with self.assertRaises(ValueError): validate_graph(g, 2)\n\n    def test_auxiliary_does_not_receive_answer(self):\n        r = row()\n        r[\'target\'][\'segments\'][0][\'reason\'] = \'SECRET_REFERENCE_TEXT\'\n        e = dict(stsg=graph(), caption={\'caption\': \'A ball rolls.\'})\n        for arm in (\'caption\', \'stsg\', \'stsg_no_links\'):\n            samples = sft_rows(r, e, arm)\n            self.assertIn(\'SECRET_REFERENCE_TEXT\', samples[0][\'messages\'][1][\'content\'])\n            self.assertNotIn(\'SECRET_REFERENCE_TEXT\', json.dumps(samples[1]))\n            self.assertEqual(len(samples), 2)\n        self.assertEqual(graph(), e[\'stsg\'])\n        no_links = json.loads(sft_rows(r, e, \'stsg_no_links\')[1][\'messages\'][1][\'content\'])\n        self.assertEqual(no_links[\'temporal_links\'], [])\n        self.assertEqual(no_links[\'events\'], e[\'stsg\'][\'events\'])\n\n    def test_no_gt_duration_fallback(self):\n        with self.assertRaises(ValueError): annotation({\'video\': \'v.mp4\', \'conversations\': []})\n\n    def test_full_split_uses_every_non_dev_train_video(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            source = lambda i: dict(video=f\'{i}.mp4\', video_duration=2.,\n                                    conversations=[{\'from\': \'gpt\', \'value\': json.dumps(row()[\'target\'])}])\n            write_jsonl(root / \'train.jsonl\', [source(i) for i in range(6)])\n            write_jsonl(root / \'test.jsonl\', [source(99)])\n            split(SimpleNamespace(train=root / \'train.jsonl\', test=root / \'test.jsonl\',\n                                  video_root=tmp, output=root / \'split\', train_size=-1,\n                                  dev_size=2, seed=1))\n            train = read_jsonl(root / \'split/train.jsonl\')\n            dev = read_jsonl(root / \'split/dev.jsonl\')\n            self.assertEqual((len(train), len(dev)), (4, 2))\n            self.assertEqual(len({r[\'video_id\'] for r in train + dev}), 6)\n\n    def test_matching_penalizes_extra_segments(self):\n        gt = [segment()]\n        self.assertEqual(stats(gt * 2, gt)[0], 1)\n        self.assertAlmostEqual(aggregate(np.asarray([stats(gt * 2, gt)]))[\'joint_f1_at_05\'], 2/3)\n        self.assertEqual(stats([segment(emotion=\'Fear\')], gt)[0], 0)\n\n    def test_invalid_outputs_and_missing_denominator(self):\n        self.assertEqual(valid_segments({\'segments\': [segment(b=float(\'nan\'))]}, 2), [])\n        self.assertEqual(valid_segments({\'segments\': [segment(a=1.)]}, 2), [])\n        a, report = evaluate([row(\'a\'), row(\'b\')], [{\'video_id\': \'a\', \'dar\': row()[\'target\']}])\n        self.assertEqual(report[\'strict\'][\'coverage\'], .5)\n        self.assertEqual(report[\'missing_videos\'], 1)\n        self.assertEqual(report[\'strict\'][\'gt_segments\'], 2)\n        self.assertAlmostEqual(report[\'strict\'][\'joint_f1_at_05\'], 2/3)\n\n    def test_duplicate_predictions_fail(self):\n        p = {\'video_id\': \'v\', \'dar\': row()[\'target\']}\n        with self.assertRaises(ValueError): evaluate([row()], [p, p])\n\n    def test_official_functions_preserved(self):\n        f = official_functions()\n        self.assertIn(\'SILENT\', f[\'build_eval_prompt\'](2.))\n        self.assertEqual(f[\'evaluate_single_video\']([segment()], [segment()])[\'avg_iou\'], 1.)\n\n    def test_split_and_common_support(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            def source(i):\n                return dict(video=f\'{i}.mp4\', video_duration=2., conversations=[{\'from\': \'gpt\', \'value\': json.dumps(row()[\'target\'])}])\n            write_jsonl(root/\'train.jsonl\', [source(i) for i in range(6)])\n            write_jsonl(root/\'test.jsonl\', [source(99)])\n            args = SimpleNamespace(train=root/\'train.jsonl\', test=root/\'test.jsonl\', video_root=tmp,\n                                   output=root/\'split\', train_size=3, dev_size=2, seed=1)\n            split(args)\n            inputs = read_jsonl(root/\'split/teacher_inputs.jsonl\')\n            self.assertTrue(all(set(r) == {\'video_id\', \'video_path\', \'video_duration\'} for r in inputs))\n            evidence = [dict(video_id=r[\'video_id\'], input_sha256=digest(r), stsg=graph(), caption={\'caption\': \'Ball\'}) for r in inputs]\n            evidence[0].pop(\'stsg\')\n            write_jsonl(root/\'evidence.jsonl\', evidence)\n            build(SimpleNamespace(split=root/\'split\', evidence=root/\'evidence.jsonl\', output=root/\'arms\'))\n            for arm in (\'baseline\', \'caption\', \'stsg\', \'stsg_no_links\'):\n                self.assertEqual(len(read_jsonl(root/\'arms\'/f\'{arm}.jsonl\')), 4)\n            write_jsonl(root/\'overlap.jsonl\', [source(1)])\n            args.test = root/\'overlap.jsonl\'\n            args.output = root/\'bad\'\n            with self.assertRaises(ValueError): split(args)\n\n    def test_single_arm_only_requires_its_evidence(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            r = row()\n            write_jsonl(root / \'train.jsonl\', [r])\n            identity = {k: r[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            for arm, payload in [(\'stsg\', {\'stsg\': graph()}),\n                                 (\'caption\', {\'caption\': {\'caption\': \'A ball rolls.\'}})]:\n                evidence_path = root / (arm + \'-evidence.jsonl\')\n                write_jsonl(evidence_path, [dict(video_id=r[\'video_id\'], input_sha256=digest(identity), **payload)])\n                output = root / arm\n                build(SimpleNamespace(split=root, evidence=evidence_path, output=output, arms=[arm]))\n                self.assertEqual(len(read_jsonl(output / (arm + \'.jsonl\'))), 2)\n                self.assertEqual([p.name for p in output.glob(\'*.jsonl\')], [arm + \'.jsonl\'])\n                other = \'caption\' if arm == \'stsg\' else \'stsg\'\n                with self.assertRaises(ValueError):\n                    build(SimpleNamespace(split=root, evidence=evidence_path, output=root/\'invalid\', arms=[other]))\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'experiments/dar_stsg/teacher_videollama3.py': '"""Frozen VideoLLaMA3 teacher; run in a separate Transformers 4.57.1 process.\n\nAPI: https://github.com/DAMO-NLP-SG/VideoLLaMA3/blob/main/inference/example_videollama3.py\nUse a full local snapshot including custom Python files, not the Image checkpoint.\n"""\nimport hashlib\nimport importlib.metadata\nimport time\nfrom pathlib import Path\n\nfrom core import digest\n\n\nclass VideoLLaMA3Backend:\n    def __init__(self, model, frames=16, load_in_4bit=False):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig\n        from transformers import image_utils\n        if transformers.__version__ != \'4.57.1\':\n            raise RuntimeError(\n                \'Use the separate teacher runtime: transformers==4.57.1; \'\n                f\'loaded {transformers.__version__} from {transformers.__file__}. \'\n                \'Rerun the offline teacher bootstrap and use its teacher_python/teacher_env.\')\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'VideoLLaMA3 teacher requires CUDA\')\n        # The snapshot uses VideoInput only as a type annotation. Transformers\n        # 4.57.1 no longer exports it, while its ImageInput covers the same\n        # accepted image/frame values used by this processor.\n        if not hasattr(image_utils, \'VideoInput\'):\n            image_utils.VideoInput = image_utils.ImageInput\n        root = Path(model)\n        if not root.is_dir() or not list(root.glob(\'modeling*.py\')):\n            raise ValueError(\'Attach full local VideoLLaMA3 snapshot, including custom Python code\')\n        self.torch, self.frames = torch, frames\n        options = dict(trust_remote_code=True, local_files_only=True,\n                       torch_dtype=torch.float16, device_map=\'auto\', attn_implementation=\'sdpa\')\n        if load_in_4bit:\n            options[\'quantization_config\'] = BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n                llm_int8_skip_modules=[\'vision_encoder\', \'mm_projector\', \'lm_head\'])\n        else:\n            # Leave activation/KV-cache headroom on every visible GPU.\n            available = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]\n            if sum(available) < 22 * 1024**3:\n                raise RuntimeError(\'FP16 teacher needs more headroom: use two GPUs or --load-in-4bit\')\n            options[\'max_memory\'] = {i: int(n * .8) for i, n in enumerate(available)}\n        self.processor = AutoProcessor.from_pretrained(model, trust_remote_code=True, local_files_only=True)\n        self.model = AutoModelForCausalLM.from_pretrained(model, **options).eval()\n        self.metadata = dict(model_config=self.model.config.to_dict(),\n            snapshot_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest()\n                             for p in sorted(root.iterdir()) if p.suffix in (\'.py\', \'.json\')},\n            device_map={k: str(v) for k, v in getattr(self.model, \'hf_device_map\', {}).items()},\n            precision=\'nf4-fp16\' if load_in_4bit else \'fp16\',\n            sampling=\'uniform up to max_frames; native VideoLLaMA3 preprocessing\')\n        if load_in_4bit:\n            self.metadata[\'bitsandbytes\'] = importlib.metadata.version(\'bitsandbytes\')\n\n    def generate(self, row, prompt, max_tokens):\n        import numpy as np\n        from decord import VideoReader, cpu\n        if not Path(row[\'video_path\']).is_file():\n            raise FileNotFoundError(row[\'video_path\'])\n        reader = VideoReader(row[\'video_path\'], ctx=cpu(0), num_threads=2)\n        fps = float(reader.get_avg_fps())\n        if len(reader) < 1 or not np.isfinite(fps) or fps <= 0:\n            raise RuntimeError(\'Video has no frames or invalid FPS\')\n        indices = np.linspace(0, len(reader) - 1, min(self.frames, len(reader)), dtype=int)\n        frames = reader.get_batch(indices).asnumpy().transpose(0, 3, 1, 2)\n        timestamps = (indices / fps).tolist()\n        conversation = [{\'role\': \'user\', \'content\': [\n            {\'type\': \'video\', \'video\': list(frames), \'num_frames\': len(frames),\n             \'timestamps\': timestamps},\n            {\'type\': \'text\', \'text\': prompt}]}]\n        inputs = self.processor(conversation=conversation, add_system_prompt=True,\n                                add_generation_prompt=True, return_tensors=\'pt\')\n        pixel_hash = hashlib.sha256(inputs[\'pixel_values\'].contiguous().numpy().tobytes()).hexdigest()\n        device = self.model.get_input_embeddings().weight.device\n        inputs = {k: v.to(device) if isinstance(v, self.torch.Tensor) else v for k, v in inputs.items()}\n        inputs[\'pixel_values\'] = inputs[\'pixel_values\'].to(self.torch.float16)\n        self.torch.cuda.synchronize()\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)\n        self.torch.cuda.synchronize()\n        # Custom generate uses inputs_embeds: output contains generated tokens only.\n        return dict(raw=self.processor.batch_decode(output, skip_special_tokens=True)[0].strip(),\n                    generated_tokens=output.shape[1], input_tokens=inputs[\'input_ids\'].shape[1],\n                    generation_seconds=time.monotonic() - started,\n                    visual_tensor_sha256=pixel_hash, prompt_sha256=digest(prompt),\n                    max_frames=self.frames, sampling=\'uniform\', frame_indices=indices.tolist(),\n                    timestamps=timestamps, source_fps=fps,\n                    hit_token_limit=output.shape[1] >= max_tokens)\n', 'test.py': '#!/usr/bin/env python\n"""\nInference and evaluation for DAR video emotion segmentation.\n"""\n\nimport argparse\nimport json\nimport gc\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Set\n\nfrom tqdm import tqdm\n\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")\nos.environ.setdefault("VLLM_USE_V1", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "true")\n\nimport torch\nfrom vllm import LLM, SamplingParams\nfrom transformers import AutoProcessor\n\nREPO_ROOT = Path(__file__).resolve().parent\nLOCAL_QWEN_VL_UTILS = REPO_ROOT / "qwen-vl-utils" / "src"\nif LOCAL_QWEN_VL_UTILS.exists():\n    sys.path.insert(0, str(LOCAL_QWEN_VL_UTILS))\nfrom qwen_vl_utils import process_vision_info\n\nEMOTION_CANDIDATES = [\n    "Awkwardness", "Empathic Pain", "Fear", "Anger", "Sadness", "Relief",\n    "Boredom", "Joy", "Aesthetic Appreciation", "Adoration", "Admiration",\n    "Amusement", "Satisfaction", "Disgust", "Sexual Desire", "Confusion",\n    "Romance", "Craving", "Horror", "Excitement", "Nostalgia",\n    "Awe (or Wonder)", "Interest", "Calmness", "Surprise", "Entrancement", "Anxiety"\n]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model-path", default=os.environ.get("MODEL_PATH", "/path/to/DAR-R1"))\n    parser.add_argument("--test-jsonl", default=os.environ.get("TEST_JSONL", "/path/to/DAR/test.jsonl"))\n    parser.add_argument("--output-jsonl", default=os.environ.get("OUT_JSONL", "/path/to/outputs/dar_r1_test_predictions.jsonl"))\n    parser.add_argument("--video-root", default=os.environ.get("VIDEO_ROOT", "/path/to/DAR/videos"))\n    parser.add_argument("--video-path-prefix", default=os.environ.get("VIDEO_PATH_PREFIX", ""))\n    parser.add_argument("--batch-size", type=int, default=int(os.environ.get("BATCH_SIZE", "4")))\n    parser.add_argument("--max-tokens", type=int, default=int(os.environ.get("MAX_TOKENS", "4096")))\n    parser.add_argument("--max-model-len", type=int, default=int(os.environ.get("MAX_MODEL_LEN", "16384")))\n    parser.add_argument("--gpu-memory-utilization", type=float, default=float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.8")))\n    parser.add_argument("--seed", type=int, default=int(os.environ.get("SEED", "1234")))\n    return parser.parse_args()\n\n\ndef load_test_data(test_jsonl: str) -> List[Dict[str, Any]]:\n    data = []\n    with open(test_jsonl, \'r\', encoding=\'utf-8\') as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                item = json.loads(line)\n                video_path = item.get("video", "")\n                video_duration = item.get("video_duration")\n                video_id = os.path.splitext(os.path.basename(video_path))[0]\n\n                gt_segments = []\n                conversations = item.get("conversations", [])\n                if len(conversations) >= 2:\n                    gpt_text = conversations[1].get("value", "")\n                    try:\n                        gt_parsed = json.loads(gpt_text)\n                        gt_segments = gt_parsed.get("segments", [])\n                    except json.JSONDecodeError:\n                        pass\n\n                data.append({\n                    "video_id": video_id,\n                    "video_path": video_path,\n                    "video_duration": video_duration if video_duration else 30.0,\n                    "gt_segments": gt_segments\n                })\n            except json.JSONDecodeError:\n                continue\n    return data\n\n\ndef resolve_video_path(video_path: str, video_root: str, video_path_prefix: str) -> str:\n    if os.path.exists(video_path):\n        return video_path\n    if video_path_prefix and video_path.startswith(video_path_prefix):\n        return os.path.join(video_root, video_path[len(video_path_prefix):])\n    if not os.path.isabs(video_path):\n        return os.path.join(video_root, video_path)\n    return video_path\n\n\ndef load_done_ids(out_jsonl: str) -> Set[str]:\n    done = set()\n    if not os.path.exists(out_jsonl):\n        return done\n    with open(out_jsonl, "r", encoding="utf-8") as f:\n        for line in f:\n            try:\n                j = json.loads(line)\n                vid = str(j.get("video_id", ""))\n                if vid and j.get("segments"):\n                    done.add(vid)\n            except Exception:\n                continue\n    return done\n\n\ndef build_eval_prompt(video_duration: float) -> str:\n    cand_str = ", ".join(EMOTION_CANDIDATES)\n    dur_str = f"{video_duration:.1f}" if video_duration and video_duration > 0 else "unknown"\n\n    prompt = f"""You are analyzing a **SILENT video** (no audio/music/dialogue) to understand the **viewer\'s emotional journey**.\n\n## Your Task\n1. **Segment** the video into emotion phases based on when the viewer\'s dominant emotion CHANGES\n2. **Predict** the viewer\'s emotion for each segment (must be from the allowed list)\n3. **Reason** why the viewer feels that emotion\n\n## Core Principles (Based on Appraisal-Event Theory)\n- **Observation**: What visual elements do you see? (actions, objects, colors, lighting, composition)\n- **Causality**: Why would these specific visual elements trigger a particular emotion in viewers?\n- **Dynamics**: If the emotion changes from the previous segment, what specific event caused this transition?\n- **Adaptive Merging**: Emotional responses persist until a new event triggers change. Therefore:\n  - **CRITICAL: Adjacent segments MUST have DIFFERENT emotions**\n  - If two consecutive time periods evoke the same emotion, they should be MERGED into ONE segment\n  - Only create a new segment when the viewer\'s emotion genuinely CHANGES\n\n## Viewer-Centered Focus\n- This is about the **VIEWER\'s emotion** while watching, NOT the emotions of characters in the video\n- Don\'t just describe what characters feel (e.g., "the man looks happy")\n- Explain what makes the VIEWER feel a certain way\n\n## Allowed Emotions (STRICTLY choose from these 27 only):\n{cand_str}\n\n## Hard Constraints\n1. Segments MUST cover the full video continuously:\n   - First segment starts at 0.0\n   - Segments are continuous: segment[i].end_time == segment[i+1].start_time\n   - Last segment ends at {dur_str} (video duration)\n2. Timestamps: seconds with 1 decimal place\n3. **CRITICAL: Adjacent segments MUST have DIFFERENT emotions** (merge if same)\n\n## Quality Criteria for reason:\n1. **Visual Grounding**: Reference visual cues that ACTUALLY exist in the video frames. Don\'t invent details.\n2. **Causal Logic**: Your reasoning must logically connect visual evidence to emotion.\n3. **Viewer-Centeredness**: Focus on the VIEWER\'s feelings, not just describing characters\' facial expressions.\n4. **Temporal Consistency**: Your reasoning should be consistent with the historical context (previous segments).\n\nOutput ONLY a JSON object with this schema:\n{{\n  "segments": [\n    {{\n      "start_time": 0.0,\n      "end_time": <float>,\n      "emotion": "<one of the 27 emotions>",\n      "reason": "<viewer-centered, visually grounded reason>"\n    }}, ...\n  ]\n}}"""\n    return prompt\n\n\ndef make_messages(video_path: str, video_duration: float) -> List[Dict[str, Any]]:\n    abs_vp = os.path.abspath(video_path)\n    prompt_text = build_eval_prompt(video_duration)\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "video", "video": abs_vp},\n            {"type": "text", "text": prompt_text}\n        ]\n    }]\n\n\ndef build_input(processor, messages):\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    image_inputs, video_inputs, video_kwargs = process_vision_info(\n        messages,\n        image_patch_size=processor.image_processor.patch_size,\n        return_video_kwargs=True,\n        return_video_metadata=True\n    )\n    mm_data = {}\n    if image_inputs is not None:\n        mm_data[\'image\'] = image_inputs\n    if video_inputs is not None:\n        mm_data[\'video\'] = video_inputs\n    return {\'prompt\': text, \'multi_modal_data\': mm_data, \'mm_processor_kwargs\': video_kwargs}\n\n\ndef try_parse_json(text: str) -> Optional[dict]:\n    if not isinstance(text, str):\n        return None\n    s = text.strip()\n    if s.startswith("```"):\n        first_newline = s.find("\\n")\n        if first_newline != -1:\n            s = s[first_newline+1:]\n        if s.endswith("```"):\n            s = s[:-3]\n        s = s.strip()\n    l = s.find("{")\n    r = s.rfind("}")\n    if l == -1 or r == -1 or r <= l:\n        return None\n    try:\n        return json.loads(s[l:r+1])\n    except Exception:\n        return None\n\n\ndef merge_adjacent_same_emotion_segments(segments: List[Dict]) -> List[Dict]:\n    if not segments or len(segments) <= 1:\n        return segments\n    merged = []\n    current = segments[0].copy()\n    for i in range(1, len(segments)):\n        next_seg = segments[i]\n        if next_seg.get("emotion", "") == current.get("emotion", ""):\n            current["end_time"] = next_seg.get("end_time", current["end_time"])\n            cur_reason = current.get("reason", "")\n            nxt_reason = next_seg.get("reason", "")\n            if nxt_reason and nxt_reason not in cur_reason:\n                current["reason"] = cur_reason + " " + nxt_reason\n        else:\n            merged.append(current)\n            current = next_seg.copy()\n    merged.append(current)\n    return merged\n\n\ndef validate_and_fix_segments(segments: List[Dict], video_duration: float) -> List[Dict]:\n    if not segments:\n        return []\n    segments.sort(key=lambda x: x.get("start_time", 0))\n    for seg in segments:\n        emo = seg.get("emotion", "")\n        if emo not in EMOTION_CANDIDATES:\n            matched = False\n            for cand in EMOTION_CANDIDATES:\n                if cand.lower() in emo.lower() or emo.lower() in cand.lower():\n                    seg["emotion"] = cand\n                    matched = True\n                    break\n            if not matched:\n                seg["emotion"] = "Interest"\n    segments = merge_adjacent_same_emotion_segments(segments)\n    if segments[0].get("start_time", 0) != 0.0:\n        segments[0]["start_time"] = 0.0\n    if abs(segments[-1].get("end_time", 0) - video_duration) > 0.5:\n        segments[-1]["end_time"] = video_duration\n    for i in range(1, len(segments)):\n        if segments[i]["start_time"] != segments[i-1]["end_time"]:\n            segments[i]["start_time"] = segments[i-1]["end_time"]\n    return segments\n\n\ndef calculate_iou(seg1: Dict, seg2: Dict) -> float:\n    s1, e1 = seg1[\'start_time\'], seg1[\'end_time\']\n    s2, e2 = seg2[\'start_time\'], seg2[\'end_time\']\n    intersection = max(0, min(e1, e2) - max(s1, s2))\n    union = max(e1, e2) - min(s1, s2)\n    return intersection / union if union > 0 else 0.0\n\n\ndef evaluate_single_video(pred_segments: List[Dict], gt_segments: List[Dict]) -> Dict:\n    pred_count = len(pred_segments)\n    gt_count = len(gt_segments)\n    count_match = (pred_count == gt_count)\n\n    compare_count = min(pred_count, gt_count)\n\n    ious = []\n    emotion_matches = []\n    details = []\n\n    for i in range(compare_count):\n        pred_seg = pred_segments[i]\n        gt_seg = gt_segments[i]\n\n        iou = calculate_iou(pred_seg, gt_seg)\n        ious.append(iou)\n\n        pred_emo = pred_seg.get("emotion", "").strip()\n        gt_emo = gt_seg.get("emotion", "").strip()\n        emo_match = (pred_emo == gt_emo)\n        emotion_matches.append(emo_match)\n\n        details.append({\n            "segment_idx": i,\n            "pred_time": [pred_seg["start_time"], pred_seg["end_time"]],\n            "gt_time": [gt_seg["start_time"], gt_seg["end_time"]],\n            "iou": round(iou, 4),\n            "pred_emotion": pred_emo,\n            "gt_emotion": gt_emo,\n            "emotion_match": emo_match\n        })\n\n    return {\n        "count_match": count_match,\n        "pred_count": pred_count,\n        "gt_count": gt_count,\n        "compared": compare_count,\n        "avg_iou": sum(ious) / len(ious) if ious else 0.0,\n        "emotion_accuracy": sum(emotion_matches) / len(emotion_matches) if emotion_matches else 0.0,\n        "emotion_correct": sum(emotion_matches),\n        "ious": ious,\n        "emotion_matches": emotion_matches,\n        "details": details\n    }\n\n\ndef compute_overall_metrics(all_results: Dict[str, Dict]) -> Dict:\n    total_videos = len(all_results)\n    if total_videos == 0:\n        return {}\n\n    count_match_total = sum(1 for r in all_results.values() if r["count_match"])\n    all_ious = []\n    all_emotion_matches = []\n\n    for r in all_results.values():\n        all_ious.extend(r["ious"])\n        all_emotion_matches.extend(r["emotion_matches"])\n\n    return {\n        "total_videos": total_videos,\n        "segment_count_match": count_match_total,\n        "segment_count_match_rate": count_match_total / total_videos,\n        "total_compared_segments": len(all_ious),\n        "avg_iou": sum(all_ious) / len(all_ious) if all_ious else 0.0,\n        "emotion_accuracy": sum(all_emotion_matches) / len(all_emotion_matches) if all_emotion_matches else 0.0,\n        "emotion_correct": sum(all_emotion_matches),\n        "emotion_total": len(all_emotion_matches),\n    }\n\n\nif __name__ == "__main__":\n    args = parse_args()\n\n    print("=" * 80)\n    print("DAR video emotion segmentation inference and evaluation")\n    print("=" * 80)\n    print(f"Model: {args.model_path}")\n    print(f"Test JSONL: {args.test_jsonl}")\n    print(f"Output JSONL: {args.output_jsonl}")\n    print(f"Video root: {args.video_root}")\n    print(f"Batch size: {args.batch_size}")\n    print("=" * 80)\n\n    print("\\nLoading test data...")\n    test_data = load_test_data(args.test_jsonl)\n    print(f"Loaded {len(test_data)} test videos")\n\n    done_ids = load_done_ids(args.output_jsonl)\n    print(f"Completed videos in existing output: {len(done_ids)}")\n    videos_to_process = [v for v in test_data if v["video_id"] not in done_ids]\n    print(f"Videos to process: {len(videos_to_process)}")\n\n    if not videos_to_process:\n        print("\\nAll videos are already processed. Running evaluation.")\n    else:\n        print("\\nInitializing model...")\n        tensor_parallel_size = torch.cuda.device_count()\n        print(f"GPU count: {tensor_parallel_size}")\n\n        llm = LLM(\n            model=args.model_path,\n            trust_remote_code=True,\n            gpu_memory_utilization=args.gpu_memory_utilization,\n            tensor_parallel_size=tensor_parallel_size,\n            max_model_len=args.max_model_len,\n            seed=args.seed,\n        )\n        processor = AutoProcessor.from_pretrained(args.model_path)\n        sampling_params = SamplingParams(\n            temperature=0.1,\n            top_p=0.9,\n            max_tokens=args.max_tokens,\n            stop_token_ids=[],\n        )\n        print("Model initialized")\n\n        os.makedirs(os.path.dirname(args.output_jsonl) or \'.\', exist_ok=True)\n        fout = open(args.output_jsonl, "a", encoding="utf-8")\n\n        print("\\nStarting inference...")\n        success_count = 0\n        fail_count = 0\n\n        for batch_start in tqdm(range(0, len(videos_to_process), args.batch_size), desc="inference"):\n            batch_videos = videos_to_process[batch_start:batch_start + args.batch_size]\n            batch_inputs = []\n            batch_meta = []\n\n            for v in batch_videos:\n                vid = v["video_id"]\n                vp = resolve_video_path(v["video_path"], args.video_root, args.video_path_prefix)\n                if not os.path.exists(vp):\n                    result = {"video_id": vid, "video_path": vp, "segments": [], "error": "File not found"}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n                    continue\n                try:\n                    messages = make_messages(vp, v["video_duration"])\n                    inp = build_input(processor, messages)\n                    batch_inputs.append(inp)\n                    batch_meta.append(v)\n                except Exception as e:\n                    result = {"video_id": vid, "video_path": vp, "segments": [], "error": repr(e)}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n\n            if not batch_inputs:\n                continue\n\n            try:\n                outputs = llm.generate(batch_inputs, sampling_params=sampling_params)\n                for k, meta in enumerate(batch_meta):\n                    vid = meta["video_id"]\n                    duration = meta["video_duration"]\n                    try:\n                        raw_text = outputs[k].outputs[0].text\n                        parsed = try_parse_json(raw_text)\n                        if parsed and "segments" in parsed:\n                            segments = validate_and_fix_segments(parsed["segments"], duration)\n                            result = {"video_id": vid, "video_path": meta["video_path"],\n                                      "video_duration": duration, "segments": segments}\n                            success_count += 1\n                        else:\n                            result = {"video_id": vid, "video_path": meta["video_path"],\n                                      "video_duration": duration, "segments": [],\n                                      "error": "Parse failed", "raw_output": raw_text[:500]}\n                            fail_count += 1\n                    except Exception as e:\n                        result = {"video_id": vid, "video_path": meta["video_path"],\n                                  "segments": [], "error": repr(e)}\n                        fail_count += 1\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                del outputs\n                gc.collect()\n            except Exception as e:\n                for meta in batch_meta:\n                    result = {"video_id": meta["video_id"], "segments": [], "error": repr(e)}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n\n        fout.close()\n        print(f"\\nInference finished: success={success_count}, failed={fail_count}")\n\n    print("\\n" + "=" * 80)\n    print("Evaluation")\n    print("=" * 80)\n\n    gt_dict = {}\n    for item in test_data:\n        gt_dict[item["video_id"]] = item["gt_segments"]\n\n    pred_dict = {}\n    with open(args.output_jsonl, "r", encoding="utf-8") as f:\n        for line in f:\n            try:\n                j = json.loads(line)\n                vid = j.get("video_id", "")\n                segs = j.get("segments", [])\n                if vid and segs:\n                    pred_dict[vid] = segs\n            except Exception:\n                continue\n\n    print(f"GT videos: {len(gt_dict)}")\n    print(f"Predicted videos: {len(pred_dict)}")\n\n    common_ids = set(gt_dict.keys()) & set(pred_dict.keys())\n    print(f"Evaluable videos: {len(common_ids)}")\n\n    all_eval_results = {}\n    for vid in sorted(common_ids):\n        result = evaluate_single_video(pred_dict[vid], gt_dict[vid])\n        all_eval_results[vid] = result\n\n    metrics = compute_overall_metrics(all_eval_results)\n\n    print("\\n" + "=" * 80)\n    print("Evaluation Results")\n    print("=" * 80)\n    print(f"Evaluated videos:           {metrics.get(\'total_videos\', 0)}")\n    print(f"Segment count matches:      {metrics.get(\'segment_count_match\', 0)}")\n    print(f"Segment count match rate:   {metrics.get(\'segment_count_match_rate\', 0):.4f}")\n    print(f"Compared segments:          {metrics.get(\'total_compared_segments\', 0)}")\n    print(f"Mean temporal IoU:          {metrics.get(\'avg_iou\', 0):.4f}")\n    print(f"Emotion accuracy:           {metrics.get(\'emotion_accuracy\', 0):.4f} ({metrics.get(\'emotion_correct\', 0)}/{metrics.get(\'emotion_total\', 0)})")\n    print("=" * 80)\n\n    print("\\nEmotion accuracy under IoU thresholds:")\n    all_ious = []\n    all_emo_matches = []\n    for r in all_eval_results.values():\n        all_ious.extend(r["ious"])\n        all_emo_matches.extend(r["emotion_matches"])\n\n    for threshold in [0.3, 0.5, 0.7, 0.9]:\n        valid = [(iou, em) for iou, em in zip(all_ious, all_emo_matches) if iou >= threshold]\n        if valid:\n            acc = sum(em for _, em in valid) / len(valid)\n            print(f"  IOU >= {threshold:.1f}: {acc:.4f} ({sum(em for _, em in valid)}/{len(valid)} segments)")\n        else:\n            print(f"  IOU >= {threshold:.1f}: N/A (0 segments)")\n\n    print("\\nExample results (first 5 videos):")\n    for vid in sorted(common_ids)[:5]:\n        r = all_eval_results[vid]\n        print(f"\\n  video {vid}: pred={r[\'pred_count\']} gt={r[\'gt_count\']} "\n              f"count_match={r[\'count_match\']} "\n              f"avgIoU={r[\'avg_iou\']:.3f} emotion_acc={r[\'emotion_accuracy\']:.3f}")\n        for d in r["details"]:\n            print(f"    seg{d[\'segment_idx\']}: IOU={d[\'iou\']:.3f} "\n                  f"{d[\'pred_emotion\']} vs {d[\'gt_emotion\']} "\n                  f"match={d[\'emotion_match\']}")\n\n    eval_output_path = args.output_jsonl.replace(".jsonl", "_metrics.json")\n    eval_data = {\n        "summary": metrics,\n        "per_video": {vid: {k: v for k, v in r.items() if k != "details"} for vid, r in all_eval_results.items()}\n    }\n    with open(eval_output_path, "w", encoding="utf-8") as f:\n        json.dump(eval_data, f, ensure_ascii=False, indent=2)\n    print(f"\\nSaved metrics to: {eval_output_path}")\n', 'experiments/dar_step/common.py': '"""Small, dependency-free contracts shared by STEP-inspired data stages."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef dump(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    # Callers use fresh run directories. A failed write never becomes a valid cache.\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,\n                                    allow_nan=False), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\n\ndef compact(value):\n    return json.dumps(value, ensure_ascii=False, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef parse(text):\n    text = text.strip()\n    if text.startswith(\'```\') and text.endswith(\'```\'):\n        text = text.split(\'\\n\', 1)[1].rsplit(\'```\', 1)[0].strip()\n    value = json.loads(text)\n    require(isinstance(value, dict), \'Expected JSON object\')\n    return value\n\n\ndef bounded_text(value, limit=160):\n    require(isinstance(value, str) and 0 < len(value.strip()) <= limit, \'Invalid/too long text\')\n    require(\'<image>\' not in value and \'<video>\' not in value, \'Reserved media token in graph text\')\n    return value\n\n\ndef number(value):\n    require(not isinstance(value, bool) and isinstance(value, (int, float))\n            and math.isfinite(value), \'Expected finite number\')\n    return float(value)\n', 'experiments/dar_step/hecate_keyframes.py': '"""Python/OpenCV port of Hecate\'s histogram/subshot keyframe selection.\n\nDerived from Yahoo Hecate, Copyright 2016 Yahoo Inc., Apache-2.0.\nUpstream: 63d1c91eea847ac45545520274b78fdad1df32e0\nSee HECATE_NOTICE.md and HECATE_LICENSE for attribution and modifications.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom common import require\n\nHECATE_REVISION = \'63d1c91eea847ac45545520274b78fdad1df32e0\'\n\n\ndef histogram_features(bgr):\n    """hist_opencv.hpp: two levels, x-major patches, HSV128 + edge8/8.\n\n    Match source defaults (2000 dimensions), including separate L2 histogram\n    normalization, HSV range [0,256), patch-local Scharr and magnitude clipping\n    by histogram range. Input is the resized uint8 BGR frame.\n    """\n    import cv2\n    require(bgr.dtype == np.uint8 and bgr.ndim == 3 and bgr.shape[2] == 3,\n            \'Expected uint8 BGR frame\')\n    height, width = bgr.shape[:2]\n    require(min(height, width) >= 2, \'Frame too small for spatial pyramid\')\n    gray = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (3, 3), 0)\n    color, edges = [], []\n\n    def hist(a, bins, upper):\n        values = cv2.calcHist([a], [0], None, [bins], [0, upper])\n        return cv2.normalize(values, None).ravel()\n\n    for side in (1, 2):\n        w, h = width // side, height // side\n        for x in range(side):\n            for y in range(side):\n                area = np.s_[y*h:(y+1)*h, x*w:(x+1)*w]\n                hsv = cv2.cvtColor(bgr[area], cv2.COLOR_BGR2HSV)\n                color.extend(hist(channel, 128, 256) for channel in cv2.split(hsv))\n                patch = gray[area]\n                gx = cv2.Scharr(patch, cv2.CV_32F, 1, 0)\n                gy = cv2.Scharr(patch, cv2.CV_32F, 0, 1)\n                # Compute angles in float64 like upstream atan2(double,double).\n                angle = np.degrees(np.arctan2(gy.astype(np.float64), gx.astype(np.float64)))\n                angle = np.mod(angle, 180).astype(np.float32)\n                edges.extend((hist(angle, 8, 180), hist(cv2.magnitude(gx, gy), 8, 256)))\n    return np.concatenate(color + edges).astype(np.float32)\n\n\ndef stillness_costs(pictures, bounds):\n    """Two-neighbor L2 difference from video_parser.cpp, with safe endpoints.\n\n    Float subtraction corrects upstream uint8 saturation. Compare only original\n    adjacent frames within each PySceneDetect scene; endpoints use one neighbor.\n    Lower cost means more still. No optical flow or semantic motion estimator.\n    """\n    costs = np.zeros(len(pictures), dtype=np.float64)\n    for start, end in bounds:\n        differences = []\n        for i in range(start + 1, end):\n            a, b = pictures[i - 1], pictures[i]\n            differences.append(float(np.linalg.norm(a.astype(np.float64) - b.astype(np.float64)))\n                               / (a.shape[0] * a.shape[1]))\n        if differences:\n            costs[start], costs[end - 1] = differences[0], differences[-1]\n            for i in range(start + 1, end - 1):\n                costs[i] = (differences[i - start - 1] + differences[i - start]) / 2\n    return costs\n\n\ndef cluster_labels(features, scene_count, seed=1234):\n    """Hecate global k-means++: K=min(N//2, number of shots), guarded at 1."""\n    import cv2\n    x = np.ascontiguousarray(features, dtype=np.float32)\n    require(x.ndim == 2 and len(x) > 0 and np.isfinite(x).all(), \'Invalid histogram features\')\n    require(scene_count >= 1, \'Need at least one scene\')\n    k = max(1, min(len(x) // 2, scene_count))\n    # Avoid arbitrary label fragmentation when all/some feature rows coincide.\n    k = min(k, len(np.unique(x, axis=0)))\n    if k == 1:\n        return np.zeros(len(x), dtype=np.int32), k\n    cv2.setRNGSeed(int(seed))\n    _, labels, _ = cv2.kmeans(x, k, None,\n        (cv2.TERM_CRITERIA_MAX_ITER | cv2.TERM_CRITERIA_EPS, 1000, .0001),\n        1, cv2.KMEANS_PP_CENTERS)\n    return labels.ravel(), k\n\n\ndef select_subshots(labels, costs, bounds):\n    """One minimum-cost frame per contiguous label run, bounded by scenes.\n\n    Half-open intervals fix Hecate\'s inclusion of the next run\'s first frame.\n    A repeated cluster at a later time remains a distinct subshot.\n    """\n    labels, costs = np.asarray(labels), np.asarray(costs)\n    require(len(labels) == len(costs) and np.isfinite(costs).all(), \'Invalid subshot inputs\')\n    result = []\n    for scene, (start, end) in enumerate(bounds):\n        require(0 <= start < end <= len(labels), \'Invalid scene bounds\')\n        first = start\n        for stop in range(start + 1, end + 1):\n            if stop == end or labels[stop] != labels[first]:\n                pick = first + int(np.argmin(costs[first:stop]))\n                result.append(dict(scene=scene, start=first, end=stop,\n                                   cluster=int(labels[first]), index=pick,\n                                   stillness_cost=float(costs[pick])))\n                first = stop\n    return result\n', 'experiments/dar_step/HECATE_NOTICE.md': "# Hecate-derived keyframe selector\n\n`hecate_keyframes.py` is a Python/OpenCV adaptation of Yahoo Hecate:\nCopyright 2016 Yahoo Inc.; original developer Yale Song.\nLicensed under Apache License 2.0; full text in `HECATE_LICENSE`.\n\nPinned upstream revision: `63d1c91eea847ac45545520274b78fdad1df32e0`.\n\nSource mapping:\n\n- `histogram_features`: [hist_opencv.hpp](https://github.com/yahoo/hecate/blob/63d1c91eea847ac45545520274b78fdad1df32e0/include/hecate/hist_opencv.hpp),\n  `calc_pyr_color_hist`, `calc_pyr_edge_hist`; defaults from `video_parser.hpp`.\n- `stillness_costs`: [video_parser.cpp](https://github.com/yahoo/hecate/blob/63d1c91eea847ac45545520274b78fdad1df32e0/src/hecate/video_parser.cpp), `filter_transition`.\n- `cluster_labels`, `select_subshots`: same file, `filter_redundant_and_obtain_subshots`;\n  k-means settings from `include/hecate/gapstat.hpp::perform_kmeans`.\n- Local original-source snapshot: `research/thumbnail-hecate-20260929/hecate/`.\n\nModifications made 2026-09-29:\n\n- Port C++ to Python/OpenCV. Keep source histogram defaults: 128 bins for each\n  HSV channel, 8 each for edge orientation/magnitude, five pyramid regions,\n  2000 dimensions (paper describes 2220). Analyze all decoded frames at max\n  side 160px; save selected images at original resolution.\n- PySceneDetect supplies scene boundaries. Retain every frame as a candidate\n  within these scenes. Hecate's quality/transition rejection, long-shot\n  post-processing and final thumbnail ranking are outside this replacement.\n- Global k-means uses K=min(N//2, number of scenes), at least one; cap to unique\n  descriptors and fix the OpenCV seed. No SigLIP clustering, fixed per-scene K,\n  endpoint injection, gap-statistic search, or final thumbnail ranking.\n- Use float subtraction for two-neighbor stillness rather than saturating uint8\n  subtraction. Use one neighbor at scene endpoints (zero only for a singleton),\n  and never compare across a cut.\n- Use half-open contiguous-label runs: fix the original loop's inclusion of the\n  next label's first frame. Ties choose the earliest frame within the subshot.\n- Preserve one selected frame per subshot. Reject overflow of the video budget;\n  do not silently discard subshots/scenes. A single-scene video has K=1 and may\n  have just one selected frame; it then has no within-scene motion pair.\n- VILA parses every selected frame and merges all adjacent selected frames.\n  Scene descriptions use at most four evenly spaced selected frames to respect\n  the native teacher limit. Their IDs are logged in intermediate.json and calls.\n\nThis replaces only keyframe selection in the STEP-inspired pipeline. It is not\na complete replication of Hecate or the paper's thumbnail system. CPU tests do\nnot establish graph quality or bit-for-bit equivalence to the C++ binary.\n", 'experiments/dar_step/HECATE_LICENSE': 'Apache License\nVersion 2.0, January 2004\n\nhttp://www.apache.org/licenses/\n\nTERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n1. Definitions.\n"License" shall mean the terms and conditions for use, reproduction, and distribution as defined by \nSections 1 through 9 of this document.\n\n"Licensor" shall mean the copyright owner or entity authorized by the copyright owner that is granting \nthe License.\n\n"Legal Entity" shall mean the union of the acting entity and all other entities that control, are \ncontrolled by, or are under common control with that entity. For the purposes of this definition, \n"control" means (i) the power, direct or indirect, to cause the direction or management of such entity, \nwhether by contract or otherwise, or (ii) ownership of fifty percent (50%) or more of the outstanding \nshares, or (iii) beneficial ownership of such entity.\n\n"You" (or "Your") shall mean an individual or Legal Entity exercising permissions granted by this License.\n\n"Source" form shall mean the preferred form for making modifications, including but not limited to software \nsource code, documentation source, and configuration files.\n\n"Object" form shall mean any form resulting from mechanical transformation or translation of a Source form, \nincluding but not limited to compiled object code, generated documentation, and conversions to other media types.\n\n"Work" shall mean the work of authorship, whether in Source or Object form, made available under the License, \nas indicated by a copyright notice that is included in or attached to the work (an example is provided in \nthe Appendix below).\n\n"Derivative Works" shall mean any work, whether in Source or Object form, that is based on (or derived from) \nthe Work and for which the editorial revisions, annotations, elaborations, or other modifications represent, \nas a whole, an original work of authorship. For the purposes of this License, Derivative Works shall not \ninclude works that remain separable from, or merely link (or bind by name) to the interfaces of, the Work \nand Derivative Works thereof.\n\n"Contribution" shall mean any work of authorship, including the original version of the Work and any \nmodifications or additions to that Work or Derivative Works thereof, that is intentionally submitted to \nLicensor for inclusion in the Work by the copyright owner or by an individual or Legal Entity authorized to \nsubmit on behalf of the copyright owner. For the purposes of this definition, "submitted" means any form of \nelectronic, verbal, or written communication sent to the Licensor or its representatives, including but not \nlimited to communication on electronic mailing lists, source code control systems, and issue tracking systems \nthat are managed by, or on behalf of, the Licensor for the purpose of discussing and improving the Work, but \nexcluding communication that is conspicuously marked or otherwise designated in writing by the copyright owner \nas "Not a Contribution."\n\n"Contributor" shall mean Licensor and any individual or Legal Entity on behalf of whom a Contribution has been \nreceived by Licensor and subsequently incorporated within the Work.\n\n2. Grant of Copyright License. Subject to the terms and conditions of this License, each Contributor hereby \ngrants to You a perpetual, worldwide, non-exclusive, no-charge, royalty-free, irrevocable copyright license \nto reproduce, prepare Derivative Works of, publicly display, publicly perform, sublicense, and distribute \nthe Work and such Derivative Works in Source or Object form.\n\n3. Grant of Patent License. Subject to the terms and conditions of this License, each Contributor hereby \ngrants to You a perpetual, worldwide, non-exclusive, no-charge, royalty-free, irrevocable (except as stated \nin this section) patent license to make, have made, use, offer to sell, sell, import, and otherwise transfer \nthe Work, where such license applies only to those patent claims licensable by such Contributor that are \nnecessarily infringed by their Contribution(s) alone or by combination of their Contribution(s) with the Work \nto which such Contribution(s) was submitted. If You institute patent litigation against any entity (including \na cross-claim or counterclaim in a lawsuit) alleging that the Work or a Contribution incorporated within the \nWork constitutes direct or contributory patent infringement, then any patent licenses granted to You under \nthis License for that Work shall terminate as of the date such litigation is filed.\n\n4. Redistribution. You may reproduce and distribute copies of the Work or Derivative Works thereof in any \nmedium, with or without modifications, and in Source or Object form, provided that You meet the following \nconditions:\n\nYou must give any other recipients of the Work or Derivative Works a copy of this License; and\nYou must cause any modified files to carry prominent notices stating that You changed the files; and\nYou must retain, in the Source form of any Derivative Works that You distribute, all copyright, patent, \ntrademark, and attribution notices from the Source form of the Work, excluding those notices that do not \npertain to any part of the Derivative Works; and\nIf the Work includes a "NOTICE" text file as part of its distribution, then any Derivative Works that You \ndistribute must include a readable copy of the attribution notices contained within such NOTICE file, \nexcluding those notices that do not pertain to any part of the Derivative Works, in at least one of the \nfollowing places: within a NOTICE text file distributed as part of the Derivative Works; within the Source \nform or documentation, if provided along with the Derivative Works; or, within a display generated by the \nDerivative Works, if and wherever such third-party notices normally appear. The contents of the NOTICE file \nare for informational purposes only and do not modify the License. You may add Your own attribution notices \nwithin Derivative Works that You distribute, alongside or as an addendum to the NOTICE text from the Work, \nprovided that such additional attribution notices cannot be construed as modifying the License. \n\nYou may add Your own copyright statement to Your modifications and may provide additional or different \nlicense terms and conditions for use, reproduction, or distribution of Your modifications, or for any such \nDerivative Works as a whole, provided Your use, reproduction, and distribution of the Work otherwise \ncomplies with the conditions stated in this License.\n\n5. Submission of Contributions. Unless You explicitly state otherwise, any Contribution intentionally \nsubmitted for inclusion in the Work by You to the Licensor shall be under the terms and conditions of this \nLicense, without any additional terms or conditions. Notwithstanding the above, nothing herein shall \nsupersede or modify the terms of any separate license agreement you may have executed with Licensor regarding \nsuch Contributions.\n\n6. Trademarks. This License does not grant permission to use the trade names, trademarks, service marks, or \nproduct names of the Licensor, except as required for reasonable and customary use in describing the origin \nof the Work and reproducing the content of the NOTICE file.\n\n7. Disclaimer of Warranty. Unless required by applicable law or agreed to in writing, Licensor provides the \nWork (and each Contributor provides its Contributions) on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS \nOF ANY KIND, either express or implied, including, without limitation, any warranties or conditions of TITLE, \nNON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A PARTICULAR PURPOSE. You are solely responsible for \ndetermining the appropriateness of using or redistributing the Work and assume any risks associated with Your \nexercise of permissions under this License.\n\n8. Limitation of Liability. In no event and under no legal theory, whether in tort (including negligence), \ncontract, or otherwise, unless required by applicable law (such as deliberate and grossly negligent acts) \nor agreed to in writing, shall any Contributor be liable to You for damages, including any direct, indirect, \nspecial, incidental, or consequential damages of any character arising as a result of this License or out \nof the use or inability to use the Work (including but not limited to damages for loss of goodwill, work \nstoppage, computer failure or malfunction, or any and all other commercial damages or losses), even if such \nContributor has been advised of the possibility of such damages.\n\n9. Accepting Warranty or Additional Liability. While redistributing the Work or Derivative Works thereof, \nYou may choose to offer, and charge a fee for, acceptance of support, warranty, indemnity, or other \nliability obligations and/or rights consistent with this License. However, in accepting such obligations, \nYou may act only on Your own behalf and on Your sole responsibility, not on behalf of any other Contributor, \nand only if You agree to indemnify, defend, and hold each Contributor harmless for any liability incurred \nby, or claims asserted against, such Contributor by reason of your accepting any such warranty or additional \nliability.\n\nEND OF TERMS AND CONDITIONS\n', 'experiments/dar_step/frames.py': '"""PySceneDetect scenes and source-derived Hecate keyframe selection."""\nfrom __future__ import annotations\n\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom common import file_hash, require\n\n\ndef detect_scenes(path, duration, threshold=27.0, min_scene_seconds=.5):\n    from scenedetect import open_video, SceneManager\n    from scenedetect.detectors import ContentDetector\n    video = open_video(str(path), backend=\'opencv\')\n    fps = float(video.frame_rate)\n    require(fps > 0 and math.isfinite(fps), \'Invalid video fps\')\n    manager = SceneManager()\n    manager.add_detector(ContentDetector(threshold=threshold,\n                                        min_scene_len=max(1, round(min_scene_seconds * fps))))\n    decoded = manager.detect_scenes(video=video, show_progress=False)\n    require(decoded > 0, \'Video contains no decodable frames\')\n    scenes = manager.get_scene_list(start_in_scene=True)\n    bounds = [(a.get_frames(), b.get_frames()) for a, b in scenes] or [(0, decoded)]\n    require(bounds[0][0] == 0 and bounds[-1][1] == decoded, \'Incomplete scene coverage\')\n    # Frame-index timestamps assume CFR. Do not stretch the video to annotation length.\n    measured = decoded / fps\n    require(abs(measured - duration) <= max(.25, 2 / fps),\n            f\'Video/annotation duration mismatch: {measured:.3f} vs {duration}\')\n    return fps, bounds\n\n\ndef select_frames(row, out, *, threshold=27., min_scene_seconds=.5,\n                  max_frames=24, seed=1234):\n    """All frames -> Hecate histograms -> global k-means -> still subshots."""\n    import cv2\n    from PIL import Image\n    from hecate_keyframes import (HECATE_REVISION, histogram_features,\n                                  stillness_costs, cluster_labels, select_subshots)\n    duration = row[\'video_duration\']\n    fps, bounds = detect_scenes(row[\'video_path\'], duration, threshold, min_scene_seconds)\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    capture = cv2.VideoCapture(str(row[\'video_path\']))\n    pictures = []\n    try:\n        for index in range(bounds[-1][1]):\n            ok, bgr = capture.read()\n            require(ok, f\'Cannot decode frame {index}\')\n            actual = capture.get(cv2.CAP_PROP_POS_MSEC) / 1000\n            require(abs(actual - index / fps) < max(.1, 2 / fps),\n                    \'Nonuniform/unsupported timestamps; normalize video to CFR before this pilot\')\n            height, width = bgr.shape[:2]\n            scale = min(1., 160 / max(height, width))\n            if scale < 1:\n                bgr = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)\n            pictures.append(bgr)\n        features = np.stack([histogram_features(p) for p in pictures])\n        labels, k = cluster_labels(features, len(bounds), seed)\n        costs = stillness_costs(pictures, bounds)\n        subshots = select_subshots(labels, costs, bounds)\n        require(len(subshots) <= max_frames,\n                f\'Keyframe budget {max_frames} exceeded ({len(subshots)} subshots); increase budget and context together\')\n        scenes = [dict(id=f\'c{i}\', start=round(a / fps, 4), end=round(min(duration, b / fps), 4))\n                  for i, (a, b) in enumerate(bounds)]\n        frames = []\n        # Save original-resolution images, not the 160px analysis images.\n        for subshot in subshots:\n            index = subshot[\'index\']\n            capture.set(cv2.CAP_PROP_POS_FRAMES, index)\n            ok, bgr = capture.read()\n            require(ok, f\'Cannot decode selected frame {index}\')\n            path = out / f\'f{index:06d}.png\'\n            Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).save(path)\n            frames.append(dict(id=f\'f{len(frames)}\', scene=f"c{subshot[\'scene\']}", index=index,\n                               time=round(index / fps, 4), path=str(path.resolve()), sha256=file_hash(path)))\n    finally:\n        capture.release()\n    return dict(scenes=scenes, frames=frames, fps=fps, video_sha256=file_hash(row[\'video_path\']),\n                selection=dict(method=\'hecate_histogram_subshot_stillness\', revision=HECATE_REVISION,\n                               feature_dim=2000, candidate_count=len(pictures), clusters=k, seed=seed,\n                               subshots=subshots))\n', 'experiments/dar_step/graph.py': '"""Validate FSGs, merge identities within clips, and add cross-clip references."""\nfrom __future__ import annotations\n\nimport copy\nimport re\n\nfrom common import bounded_text, number, require\n\n\ndef fsg(graph):\n    require(set(graph) == {\'objects\', \'relations\', \'observation\'}, \'Invalid frame graph keys\')\n    require(isinstance(graph[\'objects\'], list) and len(graph[\'objects\']) <= 6, \'Invalid object count\')\n    ids = set()\n    for obj in graph[\'objects\']:\n        require(set(obj) == {\'id\', \'label\', \'kind\', \'attributes\'}, \'Invalid object keys\')\n        require(isinstance(obj[\'id\'], str) and re.fullmatch(r\'o\\d{1,3}\', obj[\'id\'])\n                and obj[\'id\'] not in ids, \'Invalid/duplicate object ID\')\n        ids.add(obj[\'id\'])\n        bounded_text(obj[\'label\'], 60)\n        require(obj[\'kind\'] in (\'static\', \'dynamic\'), \'Invalid object kind\')\n        require(isinstance(obj[\'attributes\'], list) and len(obj[\'attributes\']) <= 3,\n                \'Invalid attributes\')\n        for attr in obj[\'attributes\']:\n            bounded_text(attr, 80)\n    require(isinstance(graph[\'relations\'], list) and len(graph[\'relations\']) <= 6, \'Invalid relations\')\n    for edge in graph[\'relations\']:\n        require(isinstance(edge, list) and len(edge) == 3 and edge[0] in ids\n                and edge[2] in ids and edge[0] != edge[2], \'Invalid relation endpoint\')\n        bounded_text(edge[1], 80)\n    bounded_text(graph[\'observation\'])\n    return graph\n\n\ndef verified_graph(graph, verdict):\n    require(set(verdict) == {\'objects\', \'attributes\', \'relations\', \'observation\'}, \'Invalid verification keys\')\n    require(all(isinstance(verdict[k], list) for k in (\'objects\', \'attributes\', \'relations\')),\n            \'Verification fields must be lists\')\n    by_id = {o[\'id\']: o for o in graph[\'objects\']}\n    require(all(isinstance(x, str) and x in by_id for x in verdict[\'objects\']), \'Unknown verified ID\')\n    keep = set(verdict[\'objects\'])\n    attributes = set()\n    for pair in verdict[\'attributes\']:\n        require(isinstance(pair, list) and len(pair) == 2 and pair[0] in by_id\n                and type(pair[1]) is int and 0 <= pair[1] < len(by_id[pair[0]][\'attributes\']),\n                \'Unknown verified attribute\')\n        attributes.add(tuple(pair))\n    require(all(type(i) is int and 0 <= i < len(graph[\'relations\']) for i in verdict[\'relations\']),\n            \'Unknown verified relation\')\n    require(type(verdict[\'observation\']) is bool, \'Verification must be boolean\')\n    result = copy.deepcopy(graph)\n    result[\'objects\'] = [o for o in result[\'objects\'] if o[\'id\'] in keep]\n    for obj in result[\'objects\']:\n        obj[\'attributes\'] = [a for i, a in enumerate(obj[\'attributes\']) if (obj[\'id\'], i) in attributes]\n    result[\'relations\'] = [e for i, e in enumerate(graph[\'relations\'])\n                           if i in verdict[\'relations\'] and e[0] in keep and e[2] in keep]\n    result[\'observation\'] = graph[\'observation\'] if verdict[\'observation\'] else \'\'\n    return result\n\n\ndef scoped(graph, frame_id):\n    result = copy.deepcopy(graph)\n    mapping = {o[\'id\']: frame_id + \'_\' + o[\'id\'] for o in graph[\'objects\']}\n    for obj in result[\'objects\']:\n        obj[\'id\'] = mapping[obj[\'id\']]\n    result[\'relations\'] = [[mapping[a], p, mapping[b]] for a, p, b in graph[\'relations\']]\n    return result\n\n\ndef pair_result(result, left, right, bridge=False):\n    require(set(result) == {\'matches\', \'motions\'} and all(isinstance(result[k], list)\n            for k in result), \'Invalid pairing response\')\n    a = {o[\'id\']: o for o in left[\'objects\']}\n    b = {o[\'id\']: o for o in right[\'objects\']}\n    seen_a, seen_b, pairs = set(), set(), set()\n    for edge in result[\'matches\']:\n        require(isinstance(edge, list) and len(edge) == 2, \'Invalid match\')\n        x, y = edge\n        require(x in a and y in b and x not in seen_a and y not in seen_b, \'Match must be one-to-one\')\n        require(a[x][\'kind\'] == b[y][\'kind\'], \'Conflicting static/dynamic identity\')\n        seen_a.add(x)\n        seen_b.add(y)\n        pairs.add((x, y))\n    require(len(result[\'motions\']) <= 6, \'Too many motion edges\')\n    require(not bridge or not result[\'motions\'], \'Motion across a cut is unsupported\')\n    for edge in result[\'motions\']:\n        require(isinstance(edge, list) and len(edge) == 3 and (edge[0], edge[2]) in pairs,\n                \'Motion must use matched observations\')\n        require(a[edge[0]][\'kind\'] == b[edge[2]][\'kind\'] == \'dynamic\', \'Static motion is invalid\')\n        bounded_text(edge[1], 80)\n    return result\n\n\nclass IdentityGroups:\n    """Union-find that cannot collapse two distinct objects observed together."""\n    def __init__(self, members):\n        self.parent = {key: key for key in members}\n        self.scope = {key: {scope} for key, scope in members.items()}\n\n    def find(self, key):\n        if self.parent[key] != key:\n            self.parent[key] = self.find(self.parent[key])\n        return self.parent[key]\n\n    def join(self, a, b):\n        a, b = self.find(a), self.find(b)\n        if a == b:\n            return\n        require(not self.scope[a] & self.scope[b], \'Identity collision: two objects in the same frame/scene\')\n        self.parent[b] = a\n        self.scope[a] |= self.scope[b]\n\n\ndef assemble(sampling, parsed, local_pairs, bridges, descriptions):\n    frames = sampling[\'frames\']\n    objects = {o[\'id\']: (f, o) for f in frames for o in parsed[f[\'id\']][\'objects\']}\n    require(bool(objects), \'No verified objects in video\')\n    groups = IdentityGroups({oid: f[\'id\'] for oid, (f, _) in objects.items()})\n    motions = []\n    for result in local_pairs:\n        for a, b in result[\'matches\']:\n            require(objects[a][0][\'scene\'] == objects[b][0][\'scene\'], \'Local merge crosses scene\')\n            groups.join(a, b)\n        motions.extend(result[\'motions\'])\n    entities, entity_ids, observation_entity = [], {}, {}\n    for oid, (frame, obj) in objects.items():\n        root = groups.find(oid)\n        if root not in entity_ids:\n            eid = f\'e{len(entities)}\'\n            entity_ids[root] = eid\n            entities.append(dict(id=eid, scene=frame[\'scene\'], label=obj[\'label\'], kind=obj[\'kind\']))\n        observation_entity[oid] = entity_ids[root]\n    references = set()\n    cross_groups = IdentityGroups({e[\'id\']: e[\'scene\'] for e in entities})\n    for result in bridges:\n        for a, b in result[\'matches\']:\n            x, y = observation_entity[a], observation_entity[b]\n            require(objects[a][0][\'scene\'] != objects[b][0][\'scene\'], \'Reference is within one scene\')\n            cross_groups.join(x, y)\n            references.add((x, \'same_as\', y))\n    result = dict(scenes=[dict(s, description=descriptions[s[\'id\']]) for s in sampling[\'scenes\']],\n                  entities=entities, frames=[], motion_links=motions,\n                  reference_links=[list(e) for e in sorted(references)],\n                  event_links=[[a[\'id\'], \'before\', b[\'id\']]\n                               for a, b in zip(sampling[\'scenes\'], sampling[\'scenes\'][1:])])\n    for frame in frames:\n        graph = parsed[frame[\'id\']]\n        result[\'frames\'].append(dict(id=frame[\'id\'], scene=frame[\'scene\'], time=frame[\'time\'],\n            objects=[dict(id=o[\'id\'], entity=observation_entity[o[\'id\']], attributes=o[\'attributes\'])\n                     for o in graph[\'objects\']], relations=graph[\'relations\'], observation=graph[\'observation\']))\n    return result\n\n\ndef validate(graph, duration):\n    require(set(graph) == {\'scenes\', \'entities\', \'frames\', \'motion_links\', \'reference_links\', \'event_links\'},\n            \'Invalid whole-video graph keys\')\n    require(all(isinstance(x, list) for x in graph.values()), \'Graph fields must be lists\')\n    scenes, entities, observations, frame_ids = {}, {}, {}, set()\n    previous_end = 0\n    for scene in graph[\'scenes\']:\n        require(set(scene) == {\'id\', \'start\', \'end\', \'description\'}, \'Invalid scene\')\n        bounded_text(scene[\'id\'], 30)\n        require(scene[\'id\'] not in scenes, \'Duplicate scene\')\n        start, end = number(scene[\'start\']), number(scene[\'end\'])\n        require(abs(start - previous_end) <= .001 and start < end <= duration + .001, \'Scene gap/overlap/bounds\')\n        if scene[\'description\']:\n            bounded_text(scene[\'description\'])\n        else:\n            require(scene[\'description\'] == \'\', \'Invalid empty description\')\n        scenes[scene[\'id\']] = scene\n        previous_end = end\n    require(scenes and abs(previous_end - duration) <= .25, \'Scenes do not cover video\')\n    for entity in graph[\'entities\']:\n        require(set(entity) == {\'id\', \'scene\', \'label\', \'kind\'}, \'Invalid entity\')\n        bounded_text(entity[\'id\'], 30)\n        bounded_text(entity[\'label\'], 60)\n        require(entity[\'id\'] not in entities and entity[\'scene\'] in scenes\n                and entity[\'kind\'] in (\'static\', \'dynamic\'), \'Invalid/duplicate entity\')\n        entities[entity[\'id\']] = entity\n    require(entities and 0 < len(graph[\'frames\']) <= 64, \'Empty/oversize graph\')\n    previous_time, covered = -1, set()\n    for frame in graph[\'frames\']:\n        require(set(frame) == {\'id\', \'scene\', \'time\', \'objects\', \'relations\', \'observation\'}, \'Invalid frame\')\n        require(frame[\'scene\'] in scenes and frame[\'id\'] not in frame_ids, \'Invalid frame reference\')\n        frame_ids.add(frame[\'id\'])\n        covered.add(frame[\'scene\'])\n        time = number(frame[\'time\'])\n        scene = scenes[frame[\'scene\']]\n        require(previous_time < time and scene[\'start\'] <= time < scene[\'end\'], \'Invalid frame time\')\n        previous_time = time\n        local, local_entities = set(), set()\n        require(isinstance(frame[\'objects\'], list) and len(frame[\'objects\']) <= 6, \'Object count\')\n        for obj in frame[\'objects\']:\n            require(set(obj) == {\'id\', \'entity\', \'attributes\'} and obj[\'id\'] not in observations,\n                    \'Invalid/duplicate observation\')\n            require(obj[\'entity\'] in entities and entities[obj[\'entity\']][\'scene\'] == frame[\'scene\']\n                    and obj[\'entity\'] not in local_entities, \'Entity collision/scope\')\n            require(isinstance(obj[\'attributes\'], list) and len(obj[\'attributes\']) <= 3, \'Attributes\')\n            for attr in obj[\'attributes\']:\n                bounded_text(attr, 80)\n            local.add(obj[\'id\'])\n            local_entities.add(obj[\'entity\'])\n            observations[obj[\'id\']] = (frame, obj)\n        require(isinstance(frame[\'relations\'], list) and len(frame[\'relations\']) <= 6, \'Relation count\')\n        for edge in frame[\'relations\']:\n            require(isinstance(edge, list) and len(edge) == 3 and edge[0] in local and edge[2] in local\n                    and edge[0] != edge[2], \'Invalid spatial edge\')\n            bounded_text(edge[1], 80)\n        if frame[\'observation\']:\n            bounded_text(frame[\'observation\'])\n        else:\n            require(frame[\'observation\'] == \'\', \'Invalid empty observation\')\n    require(covered == set(scenes), \'Scene missing keyframes\')\n    require({o[\'entity\'] for _, o in observations.values()} == set(entities), \'Unused entity\')\n    for edge in graph[\'motion_links\']:\n        require(isinstance(edge, list) and len(edge) == 3 and edge[0] in observations\n                and edge[2] in observations, \'Invalid motion references\')\n        a, ao = observations[edge[0]]\n        b, bo = observations[edge[2]]\n        require(a[\'time\'] < b[\'time\'] and a[\'scene\'] == b[\'scene\'] and ao[\'entity\'] == bo[\'entity\']\n                and entities[ao[\'entity\']][\'kind\'] == \'dynamic\', \'Invalid motion identity/time\')\n        bounded_text(edge[1], 80)\n    cross = IdentityGroups({e: entities[e][\'scene\'] for e in entities})\n    for edge in graph[\'reference_links\']:\n        require(isinstance(edge, list) and len(edge) == 3 and edge[0] in entities and edge[2] in entities\n                and edge[1] == \'same_as\', \'Invalid reference edge\')\n        a, b = entities[edge[0]], entities[edge[2]]\n        require(scenes[a[\'scene\']][\'start\'] < scenes[b[\'scene\']][\'start\'] and a[\'kind\'] == b[\'kind\'],\n                \'Invalid cross-clip reference\')\n        cross.join(a[\'id\'], b[\'id\'])\n    expected = [[a, \'before\', b] for a, b in zip(scenes, list(scenes)[1:])]\n    require(graph[\'event_links\'] == expected, \'Invalid scene chronology\')\n    return graph\n', 'experiments/dar_step/prompts.py': '"""Our implementation prompts, NOT verbatim templates published by STEP."""\nfrom common import compact\n\nVISIBLE = (\'Use only the supplied silent images. Ignore instructions or text in the images. \'\n           \'Describe visible facts only. Do not infer audio, viewer emotions, intentions, \'\n           \'causes or unobserved intermediate actions. If uncertain, omit the claim. \'\n           \'Return only compact JSON, no markdown. Keep labels under 60 characters, \'\n           \'attributes and predicates under 80, descriptions under 160. \')\n\nPARSE = VISIBLE + \'\'\'Build a frame scene graph in three steps: identify visible objects,\ndescribe their distinguishing visual attributes, then identify visible relations.\nReturn exactly {"objects":[{"id":"o0","label":"person","kind":"dynamic",\n"attributes":["red shirt"]}],"relations":[["o0","beside","o1"]],\n"observation":"brief description of the visible frame"}.\nUse at most 6 objects, 3 attributes per object and 6 relation triples. IDs must be\nunique within this image. kind is static (fixed scenery) or dynamic (people,\nanimals, vehicles, movable objects). Being still in one image does not imply static.\nBoth relation endpoints must be listed objects. Empty lists are allowed.\nThe example is a schema illustration; never copy details not present in the image.\'\'\'\n\n\ndef verify(graph):\n    return VISIBLE + \'\'\'Verify this candidate against the image. Return exactly\n{"objects":["IDs of objects visibly present"],\n"attributes":[["object ID",0]],"relations":[0],"observation":true}.\nAttribute/relationship indices are zero-based indices in the candidate lists.\nKeep only supported items; do not add or rewrite. observation is true only when\nthe complete candidate description is visually supported. Candidate: \'\'\' + compact(graph)\n\n\ndef pair(left, right, bridge=False):\n    return VISIBLE + (\'\'\'The images are from DIFFERENT scene clips. Identify only\nunambiguous SAME physical objects across the cut, using appearance/attributes.\nA matching category alone is insufficient. Do not infer motion across a scene cut.\nReturn exactly {"matches":[["first image object ID","second image object ID"]],\n"motions":[]}. \'\'\' if bridge else \'\'\'The images are chronological keyframes in ONE\nscene clip. Identify unambiguous SAME physical objects, one-to-one, using appearance.\nReturn exactly {"matches":[["first image object ID","second image object ID"]],\n"motions":[["first image object ID","visible state/position change","second image object ID"]]}.\nMotion endpoints must also be a matched pair and both must be dynamic objects.\nDescribe only change supported by these endpoints; do not invent trajectory.\n\'\'\') + \'Empty arrays are valid. First graph: \' + compact(left) + \'\\nSecond graph: \' + compact(right)\n\n\ndef event_prompt(frames):\n    return VISIBLE + \'Summarize the visible state/change in this scene clip in at most 160 characters. \' + \\\n        \'Return exactly {"description":"..."}. Sparse frames do not show everything between them. \' + \\\n        \'Times in seconds: \' + compact([f[\'time\'] for f in frames])\n\n\nSTUDENT = \'\'\'Construct a visual spatio-temporal graph for this whole video from the\nchronological keyframes and scene intervals below. Times are absolute seconds.\nReturn JSON with exactly: scenes, entities, frames, motion_links, reference_links,\nevent_links. scenes: [{id,start,end,description}]; entities: [{id,scene,label,kind}];\nframes: [{id,scene,time,objects:[{id,entity,attributes:[string]}],\nrelations:[[object_id,predicate,object_id]],observation}].\nMerge confidently identical objects within a scene into one entity, retaining\nseparate timestamped object observations. motion_links connect dynamic object\nobservations in the same scene as [earlier_object_id,visible_change,later_object_id].\nreference_links connect the same physical entities across scenes as\n[earlier_entity_id,"same_as",later_entity_id]. event_links connect consecutive\nscene IDs as [earlier_scene_id,"before",later_scene_id]. Never infer causality.\nUse at most 6 objects and 6 relations per frame, 3 attributes per object.\nUse short labels and predicates and descriptions under 160 characters.\nOnly visible evidence; omit uncertain identities, motion and relations.\nScene cuts are not DAR emotion boundaries. No audio or viewer emotion labels.\'\'\'\n', 'experiments/dar_step/teacher_vila.py': '"""Native VILA 1.5 adapter; source revision and runtime are pinned separately."""\nfrom pathlib import Path\n\nfrom common import require\n\nVILA_REVISION = \'6b941da19e31ddfdfaa60160908ccf0978d96615\'\n\n\nclass Vila:\n    def __init__(self, model_path, device=\'cuda\'):\n        import torch\n        import transformers\n        from llava.model.builder import load_pretrained_model\n        require(transformers.__version__ == \'4.36.2\', \'VILA requires its isolated Transformers 4.36.2 runtime\')\n        require(Path(model_path).is_dir(), \'Attach a complete local VILA1.5-3b snapshot\')\n        self.torch = torch\n        self.tokenizer, self.model, self.processor, _ = load_pretrained_model(\n            str(model_path), \'VILA1.5-3b\', device=device, device_map=device,\n            attn_implementation=\'eager\', model_max_length=4096)\n        self.model.eval()\n        require(not self.model.config.s2, \'This adapter is for VILA1.5-3b, not 3b-s2\')\n        self.context = min(4096, self.model.llm.config.max_position_embeddings)\n\n    def tensors(self, pictures):\n        from llava.mm_utils import process_images\n        return process_images(pictures, self.processor, self.model.config).to(\n            self.model.device, dtype=self.model.dtype)\n\n    def embed(self, pictures):\n        # The same penultimate SigLIP patch features used by native VILA.\n        outputs = []\n        with self.torch.inference_mode():\n            for start in range(0, len(pictures), 4):\n                features = self.model.get_vision_tower()(self.tensors(pictures[start:start + 4]))\n                outputs.append(features.float().mean(dim=1).cpu())\n        return self.torch.cat(outputs).numpy()\n\n    def generate(self, pictures, prompt, max_tokens=900):\n        from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN\n        from llava.conversation import conv_templates, SeparatorStyle\n        from llava.mm_utils import tokenizer_image_token, KeywordsStoppingCriteria\n        require(0 < len(pictures) <= 4, \'VILA calls use at most four keyframes\')\n        # Explicit --conv-mode from the official VILA1.5-3B README example.\n        conversation = conv_templates[\'vicuna_v1\'].copy()\n        conversation.append_message(conversation.roles[0], (DEFAULT_IMAGE_TOKEN + \'\\n\') * len(pictures) + prompt)\n        conversation.append_message(conversation.roles[1], None)\n        text = conversation.get_prompt()\n        ids = tokenizer_image_token(text, self.tokenizer, IMAGE_TOKEN_INDEX,\n                                    return_tensors=\'pt\').unsqueeze(0).to(self.model.device)\n        pixels = self.tensors(pictures)\n        with self.torch.inference_mode():\n            # Check exact image-token expansion before native code can truncate it.\n            visual_tokens = self.model.encode_images(pixels).shape[1]\n        expanded = ids.shape[1] + len(pictures) * (visual_tokens - 1)\n        require(expanded + max_tokens <= self.context,\n                f\'Teacher context exceeded: {expanded}+{max_tokens}>{self.context}\')\n        stop = conversation.sep if conversation.sep_style != SeparatorStyle.TWO else conversation.sep2\n        with self.torch.inference_mode():\n            result = self.model.generate(ids, images=[pixels], do_sample=False,\n                num_beams=1, max_new_tokens=max_tokens, use_cache=True,\n                stopping_criteria=[KeywordsStoppingCriteria([stop], self.tokenizer, ids)],\n                pad_token_id=self.tokenizer.eos_token_id)\n        # Native VILA uses inputs_embeds. Transformers 4.36.2 prepends its dummy\n        # BOS (not the text prompt) to the generated IDs; exclude it from counts.\n        require(int(result[0, 0]) == self.tokenizer.bos_token_id, \'Unexpected native generation prefix\')\n        completion = result[:, 1:]\n        raw = self.tokenizer.batch_decode(completion, skip_special_tokens=True)[0].strip()\n        if raw.endswith(stop):\n            raw = raw[:-len(stop)].strip()\n        return dict(raw=raw, generated_tokens=int(completion.shape[1]), input_tokens=int(expanded),\n                    hit_token_limit=bool(completion.shape[1] >= max_tokens))\n', 'experiments/dar_step/pipeline.py': '"""Run scene -> FSG -> temporal merge -> cross-clip bridge, on train inputs only."""\nfrom __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nfrom pathlib import Path\nimport time\n\nfrom common import compact, digest, dump, file_hash, parse, read_jsonl, require\nfrom frames import select_frames\nfrom graph import assemble, fsg, pair_result, scoped, validate, verified_graph\nimport prompts\n\n\nclass Calls:\n    def __init__(self, teacher, folder):\n        self.teacher, self.folder, self.count = teacher, Path(folder), 0\n\n    def __call__(self, frames, prompt, stage, max_tokens=900):\n        from PIL import Image\n        pictures = []\n        for frame in frames:\n            with Image.open(frame[\'path\']) as image:\n                pictures.append(image.convert(\'RGB\'))\n        start = time.monotonic()\n        output = self.teacher.generate(pictures, prompt, max_tokens=max_tokens)\n        self.count += 1\n        dump(self.folder / f\'{self.count:04d}-{stage}.json\', dict(\n            stage=stage, frame_ids=[f[\'id\'] for f in frames],\n            image_sha256=[f[\'sha256\'] for f in frames], prompt=prompt,\n            seconds=time.monotonic() - start, **output))\n        require(not output[\'hit_token_limit\'], \'Teacher output hit token limit; inspect saved call\')\n        return parse(output[\'raw\'])\n\n\ndef create_graph(sampling, ask):\n    parsed, local_pairs, bridges, descriptions = {}, [], [], {}\n    for frame in sampling[\'frames\']:\n        graph = fsg(ask([frame], prompts.PARSE, \'parse\'))\n        verdict = ask([frame], prompts.verify(graph), \'verify\', max_tokens=500)\n        parsed[frame[\'id\']] = scoped(verified_graph(graph, verdict), frame[\'id\'])\n    scene_frames = {s[\'id\']: [f for f in sampling[\'frames\'] if f[\'scene\'] == s[\'id\']]\n                    for s in sampling[\'scenes\']}\n    for scene in sampling[\'scenes\']:\n        frames = scene_frames[scene[\'id\']]\n        for a, b in zip(frames, frames[1:]):\n            left, right = parsed[a[\'id\']], parsed[b[\'id\']]\n            if left[\'objects\'] and right[\'objects\']:\n                result = ask([a, b], prompts.pair(left, right), \'merge\', max_tokens=500)\n                local_pairs.append(pair_result(result, left, right))\n        event_frames = frames if len(frames) <= 4 else [frames[i * (len(frames)-1) // 3] for i in range(4)]\n        result = ask(event_frames, prompts.event_prompt(event_frames), \'event\', max_tokens=160)\n        from common import bounded_text\n        require(set(result) == {\'description\'}, \'Invalid scene event response\')\n        bounded_text(result[\'description\'])\n        verdict = ask(event_frames, prompts.VISIBLE + \'Is every claim in this candidate scene description \'\n                      \'supported by these images? Return exactly {"supported":true} or \'\n                      \'{"supported":false}. Candidate: \' + compact(result), \'verify-event\', max_tokens=40)\n        require(set(verdict) == {\'supported\'} and type(verdict[\'supported\']) is bool, \'Invalid event verdict\')\n        descriptions[scene[\'id\']] = result[\'description\'] if verdict[\'supported\'] else \'\'\n    # Build local identities first. One representative observation per local entity;\n    # group them by frame to avoid querying the same image pair repeatedly.\n    local = assemble(sampling, parsed, local_pairs, [], descriptions)\n    representative, representatives_by_scene = {}, {s[\'id\']: {} for s in sampling[\'scenes\']}\n    frame_by_id = {f[\'id\']: f for f in sampling[\'frames\']}\n    for frame in local[\'frames\']:\n        for obj in frame[\'objects\']:\n            if obj[\'entity\'] not in representative:\n                representative[obj[\'entity\']] = obj[\'id\']\n                representatives_by_scene[frame[\'scene\']].setdefault(frame[\'id\'], set()).add(obj[\'id\'])\n    for earlier, later in itertools.combinations(sampling[\'scenes\'], 2):\n        for aid, a_objects in representatives_by_scene[earlier[\'id\']].items():\n            for bid, b_objects in representatives_by_scene[later[\'id\']].items():\n                def subset(fid, ids):\n                    return dict(objects=[o for o in parsed[fid][\'objects\'] if o[\'id\'] in ids])\n                left, right = subset(aid, a_objects), subset(bid, b_objects)\n                result = ask([frame_by_id[aid], frame_by_id[bid]],\n                             prompts.pair(left, right, bridge=True), \'bridge\', max_tokens=300)\n                bridges.append(pair_result(result, left, right, bridge=True))\n    graph = assemble(sampling, parsed, local_pairs, bridges, descriptions)\n    return graph, dict(frame_graphs=parsed, local_pairs=local_pairs, bridges=bridges,\n                       event_frame_ids={sid: [f[\'id\'] for f in (fs if len(fs) <= 4 else [fs[i*(len(fs)-1)//3] for i in range(4)])]\n                                        for sid, fs in scene_frames.items()},\n                       clip_graphs=[dict(scene=s, entities=[e for e in graph[\'entities\'] if e[\'scene\'] == s[\'id\']],\n                                        frames=[f for f in graph[\'frames\'] if f[\'scene\'] == s[\'id\']],\n                                        motion_links=[m for m in graph[\'motion_links\']\n                                                      if any(m[0] == o[\'id\'] for f in graph[\'frames\']\n                                                             if f[\'scene\'] == s[\'id\'] for o in f[\'objects\'])])\n                                    for s in graph[\'scenes\']])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for key in (\'input\', \'output\', \'model\'):\n        parser.add_argument(\'--\' + key, required=True)\n    parser.add_argument(\'--threshold\', type=float, default=27.)\n    parser.add_argument(\'--min-scene-seconds\', type=float, default=.5)\n    parser.add_argument(\'--max-frames\', type=int, default=24)\n    parser.add_argument(\'--seed\', type=int, default=1234)\n    args = parser.parse_args()\n    rows = read_jsonl(args.input)\n    require(rows and len({r[\'video_id\'] for r in rows}) == len(rows), \'Empty/duplicate teacher IDs\')\n    require(all(set(r) == {\'video_id\', \'video_path\', \'video_duration\'} for r in rows),\n            \'Teacher input must be label-free: ID, path, duration only\')\n    require(1 <= args.max_frames <= 64, \'max_frames must be 1..64\')\n    # One fresh directory per run. Resume is intentionally not implicit.\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    import torch\n    import transformers\n    from teacher_vila import Vila, VILA_REVISION\n    torch.manual_seed(args.seed)\n    config = dict(vars(args), input_sha256=digest(rows), vila_revision=VILA_REVISION,\n                  torch=torch.__version__, transformers=transformers.__version__,\n                  source_sha256={p.name: file_hash(p) for p in Path(__file__).parent.glob(\'*.py\')},\n                  model_configs={str(p.relative_to(args.model)): file_hash(p)\n                                 for p in Path(args.model).rglob(\'*.json\')},\n                  sampling=\'CFR; all frames; Hecate HSV/edge histograms; global k-means; subshot stillness\',\n                  verification=\'one visual verification pass; no self-consistency sampling\')\n    dump(out / \'run.json\', config)\n    teacher = Vila(args.model)\n    with (out / \'evidence.jsonl\').open(\'x\', encoding=\'utf-8\') as stream:\n        for row in rows:\n            folder = out / digest(row[\'video_id\'])[:16]\n            folder.mkdir()\n            entry = dict(video_id=row[\'video_id\'], input_sha256=digest(row), config_sha256=digest(config))\n            calls = Calls(teacher, folder / \'calls\')\n            try:\n                sampling = select_frames(row, folder / \'frames\',\n                    threshold=args.threshold, min_scene_seconds=args.min_scene_seconds,\n                    max_frames=args.max_frames, seed=args.seed)\n                dump(folder / \'sampling.json\', sampling)\n                graph, intermediate = create_graph(sampling, calls)\n                validate(graph, row[\'video_duration\'])\n                dump(folder / \'intermediate.json\', intermediate)\n                dump(folder / \'graph.json\', graph)\n                entry.update(graph=graph, sampling=sampling)\n            except (ValueError, KeyError, TypeError) as exc:\n                entry[\'error\'] = f\'{type(exc).__name__}: {exc}\'\n                dump(folder / \'failure.json\', entry)\n            # Decoder/CUDA/infrastructure failures propagate rather than becoming data rejections.\n            entry[\'calls\'] = calls.count\n            stream.write(compact(entry) + \'\\n\')\n            stream.flush()\n            print(row[\'video_id\'], \'error=\', entry.get(\'error\'), \'calls=\', calls.count, flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_step/prepare.py': '"""Build DAR + whole-video graph SFT with the exact teacher keyframes."""\nimport argparse\nimport copy\nimport json\nfrom pathlib import Path\nimport sys\n\nfrom common import compact, digest, dump, file_hash, read_jsonl, require\nfrom graph import validate\nfrom prompts import STUDENT\n\nLEGACY = Path(__file__).resolve().parents[1] / \'dar_stsg\'\nsys.path.insert(0, str(LEGACY))\nfrom core import sft_rows as legacy_rows\n\n\ndef sft_rows(row, evidence, arm=\'stsg\'):\n    answer = legacy_rows(row, {}, \'baseline\')[0]\n    graph = copy.deepcopy(evidence[\'graph\'])\n    prompt = STUDENT\n    if arm == \'stsg_no_links\':\n        graph[\'motion_links\'], graph[\'reference_links\'], graph[\'event_links\'] = [], [], []\n        prompt += \'\\nFor this ablation leave motion_links, reference_links and event_links empty.\'\n    frames = evidence[\'sampling\'][\'frames\']\n    prompt += \'\\nScene intervals: \' + compact(evidence[\'sampling\'][\'scenes\']) + \'\\n\'\n    prompt += \'\\n\'.join(f"{f[\'id\']}, {f[\'scene\']}, t={f[\'time\']:.4f}s: <image>" for f in frames)\n    aux = dict(id=row[\'video_id\'] + \':\' + arm, images=[f[\'path\'] for f in frames], messages=[\n        dict(role=\'user\', content=prompt), dict(role=\'assistant\', content=compact(graph))])\n    return [answer, aux]\n\n\ndef build(split, evidence_path, output, arm):\n    rows = read_jsonl(Path(split) / \'train.jsonl\')\n    evidence = read_jsonl(evidence_path)\n    by_id = {e[\'video_id\']: e for e in evidence}\n    require(len(by_id) == len(evidence) and set(by_id) == {r[\'video_id\'] for r in rows},\n            \'Evidence must cover exactly all frozen train IDs, including failures\')\n    require(len({e[\'config_sha256\'] for e in evidence}) == 1, \'Mixed teacher runs\')\n    accepted, rejected, dataset = [], [], []\n    for row in rows:\n        e = by_id[row[\'video_id\']]\n        try:\n            require(\'error\' not in e, e.get(\'error\', \'Teacher error\'))\n            identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            require(e[\'input_sha256\'] == digest(identity), \'Teacher input mismatch\')\n            validate(e[\'graph\'], row[\'video_duration\'])\n            require(e[\'sampling\'][\'video_sha256\'] == file_hash(row[\'video_path\']), \'Source video changed\')\n            expected = [(f[\'id\'], f[\'scene\'], f[\'time\']) for f in e[\'graph\'][\'frames\']]\n            sampled = [(f[\'id\'], f[\'scene\'], f[\'time\']) for f in e[\'sampling\'][\'frames\']]\n            require(expected == sampled, \'Student/teacher frames or timestamps differ\')\n            intervals = [{k: s[k] for k in (\'id\', \'start\', \'end\')} for s in e[\'graph\'][\'scenes\']]\n            require(intervals == e[\'sampling\'][\'scenes\'], \'Student/teacher scene intervals differ\')\n            for frame in e[\'sampling\'][\'frames\']:\n                require(file_hash(frame[\'path\']) == frame[\'sha256\'], \'Teacher keyframe changed\')\n            dataset.extend(sft_rows(row, e, arm))\n            accepted.append(row[\'video_id\'])\n        except (ValueError, KeyError, TypeError) as exc:\n            rejected.append(dict(video_id=row[\'video_id\'], error=str(exc)))\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    dump(out / \'build.json\', dict(accepted=len(accepted), ids=accepted, rejected=rejected,\n         source_count=len(rows), arm=arm, evidence_sha256=digest(evidence),\n         dataset_sha256=digest(dataset), note=\'Graph target only during training; images exactly match teacher selection.\'))\n    require(accepted, \'No accepted graph; inspect teacher calls and build.json\')\n    (out / (arm + \'.jsonl\')).write_text(\'\'.join(compact(r) + \'\\n\' for r in dataset), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    for key in (\'split\', \'evidence\', \'output\'):\n        parser.add_argument(\'--\' + key, required=True)\n    parser.add_argument(\'--arm\', choices=(\'stsg\', \'stsg_no_links\'), default=\'stsg\')\n    args = parser.parse_args()\n    build(args.split, args.evidence, args.output, args.arm)\n', 'experiments/dar_step/preflight_sft.py': '"""Encode EVERY row with the actual ms-swift template before expensive SFT."""\nimport argparse\nimport copy\nimport os\n\nfrom common import digest, dump, read_jsonl, require\n\n# These settings must match train.sh. Irregular keyframes are multi-image inputs.\nVISION_ENV = dict(FPS_MIN_FRAMES=\'16\', FPS_MAX_FRAMES=\'16\', VIDEO_MIN_PIXELS=\'100352\',\n                  VIDEO_MAX_PIXELS=\'100352\', VIDEO_MIN_TOKEN_NUM=\'128\', VIDEO_MAX_TOKEN_NUM=\'128\',\n                  MIN_PIXELS=\'100352\', MAX_PIXELS=\'100352\', IMAGE_MIN_TOKEN_NUM=\'128\', IMAGE_MAX_TOKEN_NUM=\'128\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for key in (\'model\', \'dataset\', \'output\'):\n        parser.add_argument(\'--\' + key, required=True)\n    parser.add_argument(\'--max-length\', type=int, default=16384)\n    args = parser.parse_args()\n    os.environ.update(VISION_ENV)\n    import swift\n    from swift.llm import get_model_tokenizer, get_template\n    require(swift.__version__ == \'3.12.5\', \'Use the pinned student runtime\')\n    _, processor = get_model_tokenizer(args.model, model_type=\'qwen2_5_vl\',\n                                      load_model=False, download_model=False)\n    template = get_template(\'qwen2_5_vl\', processor, max_length=None)\n    template.set_mode(\'train\')\n    rows, lengths = read_jsonl(args.dataset), []\n    for row in rows:\n        encoded = template.encode(copy.deepcopy(row))\n        length = len(encoded[\'input_ids\'])\n        require(any(x != -100 for x in encoded[\'labels\']), \'Target missing from loss\')\n        if row.get(\'images\'):\n            require(encoded[\'image_grid_thw\'].shape[0] == len(row[\'images\']), \'Image count differs\')\n        lengths.append(dict(id=row[\'id\'], tokens=length))\n        print(row[\'id\'], length, flush=True)\n    overflow = [x for x in lengths if x[\'tokens\'] > args.max_length]\n    dump(args.output, dict(dataset_sha256=digest(rows), max_length=args.max_length,\n                          vision_env=VISION_ENV, lengths=lengths, overflow=overflow))\n    require(not overflow, f\'{len(overflow)} rows exceed context. Adjust graph/frame budget explicitly; no silent truncation.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_step/train.sh': '#!/usr/bin/env bash\n# Same DAR full SFT settings; exact multimodal length check before loading weights.\nset -euo pipefail\n: "${BASE_MODEL:?Set BASE_MODEL}"\n: "${DATA_DIR:?Set DATA_DIR}"\n: "${OUTPUT_ROOT:?Set OUTPUT_ROOT}"\nARM="${ARM:-stsg}"\ncase "$ARM" in stsg|stsg_no_links) ;; *) exit 2 ;; esac\nSCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"\nexport CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES:-0}"\nexport FPS_MIN_FRAMES=16 FPS_MAX_FRAMES=16 VIDEO_MIN_PIXELS=100352 VIDEO_MAX_PIXELS=100352\nexport VIDEO_MIN_TOKEN_NUM=128 VIDEO_MAX_TOKEN_NUM=128\nexport MIN_PIXELS=100352 MAX_PIXELS=100352 TOKENIZERS_PARALLELISM=false\nexport IMAGE_MIN_TOKEN_NUM=128 IMAGE_MAX_TOKEN_NUM=128\nSEED="${SEED:-1234}"\nMAX_LENGTH="${MAX_LENGTH:-16384}"\nDEST="$OUTPUT_ROOT/$ARM-seed$SEED"\nif [[ -e "$DEST" ]]; then echo "Output already exists: $DEST" >&2; exit 2; fi\npython "$SCRIPT_DIR/preflight_sft.py" --model "$BASE_MODEL" \\\n  --dataset "$DATA_DIR/$ARM.jsonl" --output "$DATA_DIR/$ARM-lengths.json" --max-length "$MAX_LENGTH"\npython -m swift.cli.sft \\\n  --model "$BASE_MODEL" --model_type qwen2_5_vl --template qwen2_5_vl \\\n  --dataset "$DATA_DIR/$ARM.jsonl" --split_dataset_ratio 0 \\\n  --train_type full --freeze_vit true --freeze_llm false --freeze_aligner false \\\n  --torch_dtype bfloat16 --fp16 false --bf16 true --attn_impl sdpa \\\n  --max_length "$MAX_LENGTH" --truncation_strategy delete --strict true \\\n  --num_train_epochs "${NUM_TRAIN_EPOCHS:-0.5}" --learning_rate "${LEARNING_RATE:-1e-5}" \\\n  --optim adamw_torch --weight_decay 0 \\\n  --per_device_train_batch_size 1 --gradient_accumulation_steps "${GRADIENT_ACCUMULATION_STEPS:-32}" \\\n  --gradient_checkpointing true --warmup_ratio 0.03 --lr_scheduler_type cosine \\\n  --seed "$SEED" --data_seed "$SEED" --dataloader_num_workers 0 \\\n  --save_strategy epoch --save_total_limit 1 --save_only_model true \\\n  --logging_steps 1 --report_to none --output_dir "$DEST"\n', 'experiments/dar_step/test_step.py': '"""CPU contract/integration tests; fake responses never certify teacher quality."""\nimport copy\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nimport numpy as np\n\nfrom common import digest, file_hash\nfrom frames import select_frames\nfrom hecate_keyframes import histogram_features, stillness_costs, cluster_labels, select_subshots\nfrom graph import assemble, fsg, pair_result, scoped, validate, verified_graph, IdentityGroups\nfrom pipeline import create_graph\n# Avoid colliding with the original prepare.py if both test suites share a process.\nimport importlib.util\nspec = importlib.util.spec_from_file_location(\'step_prepare\', Path(__file__).with_name(\'prepare.py\'))\nstep_prepare = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(step_prepare)\n\n\ndef frame_graph():\n    return dict(objects=[dict(id=\'o0\', label=\'person\', kind=\'dynamic\', attributes=[\'red shirt\']),\n                         dict(id=\'o1\', label=\'wall\', kind=\'static\', attributes=[\'white\'])],\n                relations=[[\'o0\', \'in front of\', \'o1\']], observation=\'A person stands by a wall.\')\n\n\ndef fixture():\n    frames = [dict(id=f\'f{i}\', scene=f\'c{i // 2}\', time=i * .5, path=f\'f{i}.png\', sha256=\'fake\')\n              for i in range(4)]\n    sampling = dict(scenes=[dict(id=\'c0\', start=0., end=1.), dict(id=\'c1\', start=1., end=2.)], frames=frames)\n    parsed = {f[\'id\']: scoped(frame_graph(), f[\'id\']) for f in frames}\n    pairs = [dict(matches=[[f\'f{a}_o0\', f\'f{b}_o0\'], [f\'f{a}_o1\', f\'f{b}_o1\']],\n                  motions=[[f\'f{a}_o0\', \'moves right\', f\'f{b}_o0\']]) for a, b in ((0, 1), (2, 3))]\n    bridges = [dict(matches=[[\'f0_o0\', \'f2_o0\'], [\'f0_o1\', \'f2_o1\']], motions=[])]\n    descriptions = {\'c0\': \'Person moves right.\', \'c1\': \'Person beside wall.\'}\n    return sampling, parsed, pairs, bridges, descriptions\n\n\nclass GraphTests(unittest.TestCase):\n    def test_merge_preserves_temporal_attributes_and_clip_identity(self):\n        args = fixture()\n        args[1][\'f1\'][\'objects\'][0][\'attributes\'] = [\'arm raised\']\n        graph = validate(assemble(*args), 2.)\n        self.assertEqual(len(graph[\'entities\']), 4)\n        self.assertEqual(len(graph[\'frames\']), 4)\n        self.assertEqual(len(graph[\'motion_links\']), 2)\n        self.assertEqual(len(graph[\'reference_links\']), 2)\n        self.assertEqual(graph[\'frames\'][1][\'objects\'][0][\'attributes\'], [\'arm raised\'])\n        self.assertEqual(graph[\'event_links\'], [[\'c0\', \'before\', \'c1\']])\n\n    def test_verification_removes_attributes_and_dangling_edges(self):\n        result = verified_graph(fsg(frame_graph()), dict(objects=[\'o0\'], attributes=[],\n                                                          relations=[0], observation=False))\n        self.assertEqual(result[\'relations\'], [])\n        self.assertEqual(result[\'objects\'][0][\'attributes\'], [])\n        self.assertEqual(result[\'observation\'], \'\')\n\n    def test_no_matching_by_label_without_visual_match(self):\n        args = list(fixture())\n        args[2], args[3] = [], []\n        self.assertEqual(len(assemble(*args)[\'entities\']), 8)\n\n    def test_collision_rejected_transitively(self):\n        union = IdentityGroups({\'a\': \'f0\', \'b\': \'f1\', \'c\': \'f0\'})\n        union.join(\'a\', \'b\')\n        with self.assertRaisesRegex(ValueError, \'collision\'):\n            union.join(\'b\', \'c\')\n\n    def test_invalid_motion_static_and_cross_cut(self):\n        left, right = scoped(frame_graph(), \'f0\'), scoped(frame_graph(), \'f1\')\n        for result, bridge in ((dict(matches=[[\'f0_o1\', \'f1_o1\']],\n                                    motions=[[\'f0_o1\', \'moves\', \'f1_o1\']]), False),\n                               (dict(matches=[[\'f0_o0\', \'f1_o0\']],\n                                     motions=[[\'f0_o0\', \'moves\', \'f1_o0\']]), True)):\n            with self.assertRaises(ValueError):\n                pair_result(result, left, right, bridge=bridge)\n\n    def test_schema_catches_invalid_times_and_references(self):\n        for mutate in (lambda g: g[\'frames\'][1].update(time=-1),\n                       lambda g: g[\'motion_links\'].append([\'f0_o0\', \'moves\', \'f2_o0\']),\n                       lambda g: g[\'reference_links\'].append([\'e0\', \'same_as\', \'e1\'])):\n            graph = assemble(*fixture())\n            mutate(graph)\n            with self.assertRaises(ValueError):\n                validate(graph, 2.)\n\n    def test_aux_uses_teacher_images_and_no_target_leak(self):\n        sampling, *rest = fixture()\n        evidence = dict(graph=assemble(sampling, *rest), sampling=sampling)\n        row = dict(video_id=\'v\', video_path=\'v.mp4\', video_duration=2.,\n                   target={\'segments\': [{\'emotion\': \'UNIQUE_GT_MARKER\', \'start\': .1, \'end\': 1.1}]})\n        dar, aux = step_prepare.sft_rows(row, evidence)\n        self.assertEqual(aux[\'images\'], [f[\'path\'] for f in sampling[\'frames\']])\n        self.assertEqual(aux[\'messages\'][0][\'content\'].count(\'<image>\'), 4)\n        self.assertNotIn(\'UNIQUE_GT_MARKER\', json.dumps(aux))\n        self.assertNotIn(\'videos\', aux)\n        self.assertEqual(dar[\'videos\'], [\'v.mp4\'])\n        self.assertEqual(json.loads(aux[\'messages\'][1][\'content\']), evidence[\'graph\'])\n        _, no_links = step_prepare.sft_rows(row, evidence, \'stsg_no_links\')\n        self.assertEqual(json.loads(no_links[\'messages\'][1][\'content\'])[\'motion_links\'], [])\n        self.assertTrue(evidence[\'graph\'][\'motion_links\'])\n\n    def test_pipeline_stage_wiring(self):\n        sampling = fixture()[0]\n        calls = []\n        def ask(frames, prompt, stage, max_tokens=900):\n            calls.append(stage)\n            if stage == \'parse\':\n                return frame_graph()\n            if stage == \'verify\':\n                return dict(objects=[\'o0\', \'o1\'], attributes=[[\'o0\', 0], [\'o1\', 0]], relations=[0], observation=True)\n            if stage == \'event\':\n                self.assertLessEqual(len(frames), 4)\n                return {\'description\': \'Person beside wall.\'}\n            if stage == \'verify-event\':\n                self.assertLessEqual(len(frames), 4)\n                return {\'supported\': True}\n            a, b = [f[\'id\'] for f in frames]\n            return dict(matches=[[a + \'_o0\', b + \'_o0\'], [a + \'_o1\', b + \'_o1\']],\n                        motions=[] if stage == \'bridge\' else [[a + \'_o0\', \'moves right\', b + \'_o0\']])\n        graph, intermediate = create_graph(sampling, ask)\n        validate(graph, 2.)\n        self.assertEqual(len(intermediate[\'clip_graphs\']), 2)\n        self.assertEqual(calls.count(\'parse\'), 4)\n        self.assertEqual(calls.count(\'merge\'), 2)\n        self.assertEqual(calls.count(\'bridge\'), 1)\n        calls.clear()\n        sampling = dict(scenes=[dict(id=\'c0\', start=0., end=3.)],\n                        frames=[dict(id=f\'f{i}\', scene=\'c0\', time=i*.5, path=f\'f{i}.png\', sha256=\'fake\')\n                                for i in range(6)])\n        graph, intermediate = create_graph(sampling, ask)\n        validate(graph, 3.)\n        self.assertEqual(calls.count(\'parse\'), 6)\n        self.assertEqual(calls.count(\'merge\'), 5)\n        self.assertEqual(intermediate[\'event_frame_ids\'][\'c0\'], [\'f0\', \'f1\', \'f3\', \'f5\'])\n\n\n    def test_build_accepts_bound_evidence_and_rejects_changed_images(self):\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            sampling, *rest = fixture()\n            video = root / \'v.mp4\'\n            video.write_bytes(b\'video content for hashing\')\n            sampling[\'video_sha256\'] = file_hash(video)\n            for frame in sampling[\'frames\']:\n                path = root / frame[\'path\']\n                path.write_bytes(frame[\'id\'].encode())\n                frame.update(path=str(path), sha256=file_hash(path))\n            identity = dict(video_id=\'v\', video_path=str(video), video_duration=2.)\n            row = dict(identity, target={\'segments\': [{\'emotion\': \'UNIQUE_GT_MARKER\'}]})\n            (root / \'train.jsonl\').write_text(json.dumps(row) + \'\\n\')\n            e = dict(video_id=\'v\', input_sha256=digest(identity), config_sha256=\'one-run\',\n                     graph=assemble(sampling, *rest), sampling=sampling)\n            evidence_path = root / \'evidence.jsonl\'\n            evidence_path.write_text(json.dumps(e) + \'\\n\')\n            step_prepare.build(root, evidence_path, root / \'accepted\', \'stsg\')\n            dataset = [json.loads(s) for s in (root / \'accepted/stsg.jsonl\').read_text().splitlines()]\n            self.assertEqual(len(dataset), 2)\n            self.assertNotIn(\'UNIQUE_GT_MARKER\', json.dumps(dataset[1]))\n            Path(sampling[\'frames\'][0][\'path\']).write_bytes(b\'changed\')\n            with self.assertRaisesRegex(ValueError, \'No accepted\'):\n                step_prepare.build(root, evidence_path, root / \'rejected\', \'stsg\')\n            report = json.loads((root / \'rejected/build.json\').read_text())\n            self.assertIn(\'keyframe changed\', report[\'rejected\'][0][\'error\'])\n\n\nclass SamplingTests(unittest.TestCase):\n    def test_histogram_source_layout(self):\n        x = histogram_features(np.zeros((32, 32, 3), np.uint8))\n        self.assertEqual(x.shape, (2000,))\n        expected = np.zeros(2000, np.float32)\n        expected[np.arange(15) * 128] = 1\n        expected[1920 + np.arange(10) * 8] = 1\n        np.testing.assert_array_equal(x, expected)\n\n    def test_clustering_determinism_and_source_cluster_count(self):\n        x = np.array([[1, 0], [1, .01], [0, 1], [.01, 1]], np.float32)\n        labels, k = cluster_labels(x, 2)\n        self.assertEqual(k, 2)\n        np.testing.assert_array_equal(labels, cluster_labels(x, 2)[0])\n        self.assertEqual(labels[0], labels[1])\n        self.assertNotEqual(labels[0], labels[2])\n        self.assertEqual(cluster_labels(x, 1)[1], 1)\n        self.assertEqual(cluster_labels(np.ones((10, 3)), 4)[1], 1)\n        self.assertEqual(cluster_labels(np.ones((1, 3)), 1)[1], 1)\n\n    def test_temporal_runs_keep_recurrence_and_new_label_boundary(self):\n        runs = select_subshots([0, 0, 1, 1, 0, 0], [4, 2, 0, 3, 8, 1], [(0, 6)])\n        self.assertEqual([r[\'index\'] for r in runs], [1, 2, 5])\n        self.assertEqual([(r[\'start\'], r[\'end\']) for r in runs], [(0, 2), (2, 4), (4, 6)])\n        self.assertEqual(len(select_subshots([0, 0], [1, 1], [(0, 1), (1, 2)])), 2)\n\n    def test_stillness_uses_float_neighbors_and_respects_cuts(self):\n        pictures = [np.full((4, 4, 3), v, np.uint8) for v in (0, 100, 100, 100, 255)]\n        cost = stillness_costs(pictures, [(0, 4), (4, 5)])\n        self.assertGreater(cost[0], 0)\n        self.assertAlmostEqual(cost[1], cost[0] / 2)\n        np.testing.assert_array_equal(cost[2:], [0, 0, 0])\n        self.assertEqual(select_subshots([0]*4, cost[:4], [(0, 4)])[0][\'index\'], 2)\n\n    def test_real_scene_detection_and_frame_extraction(self):\n        import cv2\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            for cut in (True, False):\n                path = root / f\'video-{cut}.avi\'\n                writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*\'MJPG\'), 10, (64, 64))\n                self.assertTrue(writer.isOpened())\n                for i in range(20):\n                    image = np.zeros((64, 64, 3), np.uint8)\n                    image[:] = (255, 0, 0) if i < 10 or not cut else (0, 0, 255)\n                    writer.write(image)\n                writer.release()\n                row = dict(video_id=\'v\', video_path=str(path), video_duration=2.)\n                sampling = select_frames(row, root / f\'frames-{cut}\')\n                self.assertEqual(len(sampling[\'scenes\']), 2 if cut else 1)\n                self.assertEqual(sampling[\'frames\'][0][\'index\'], 0)\n                self.assertEqual(len(sampling[\'frames\']), 2 if cut else 1)\n                self.assertEqual(sampling[\'selection\'][\'candidate_count\'], 20)\n                self.assertEqual(sampling[\'selection\'][\'feature_dim\'], 2000)\n                self.assertEqual(sampling[\'frames\'][-1][\'index\'], 10 if cut else 0)\n                self.assertEqual(sampling[\'video_sha256\'], file_hash(path))\n                for frame in sampling[\'frames\']:\n                    self.assertEqual(file_hash(frame[\'path\']), frame[\'sha256\'])\n                if cut:\n                    with self.assertRaisesRegex(ValueError, \'budget\'):\n                        select_frames(row, root / f\'budget-{cut}\', max_frames=1)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
for name, content in FILES.items():
    destination = PROJECT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_text(encoding='utf-8') == content, f'Code version mismatch: {destination}'
    else:
        destination.write_text(content, encoding='utf-8')
SCRIPTS = PROJECT / 'experiments/dar_stsg'
STEP_SCRIPTS = PROJECT / 'experiments/dar_step'
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run([teacher_python, '-m', 'unittest', 'discover', '-s', str(STEP_SCRIPTS),
                '-p', 'test_step.py', '-v'], check=True, env=teacher_env)


Tách dev từ **train**, kiểm tra overlap với test. Không dùng official test để
chọn hyperparameters. File teacher chỉ chứa video ID/path/duration, không có target.
Lỗi file/decoder/GPU sẽ dừng; không lặng lẽ thay video.

In [ ]:
run(sys.executable, SCRIPTS / 'prepare.py', 'split', '--train', TRAIN_JSONL,
    '--test', TEST_JSONL, '--video-root', VIDEO_ROOT, '--train-size', TRAIN_SIZE,
    '--dev-size', DEV_SIZE, '--output', WORK / 'split')
for name in ('teacher_inputs.jsonl', 'dev.jsonl'):
    rows = [json.loads(x) for x in (WORK / 'split' / name).read_text().splitlines()]
    missing = [r['video_path'] for r in rows if not Path(r['video_path']).is_file()]
    assert not missing, f'VIDEO_ROOT phải chứa trực tiếp các basename: {missing[:5]}'


In [ ]:
teacher_command = [teacher_python, STEP_SCRIPTS / 'pipeline.py',
    '--model', TEACHER_MODEL, '--input', WORK / 'split/teacher_inputs.jsonl',
    '--output', WORK / 'teacher', '--threshold', SCENE_THRESHOLD,
    '--min-scene-seconds', MIN_SCENE_SECONDS, '--max-frames', MAX_KEYFRAMES, '--seed', SEED]
subprocess.run(list(map(str, teacher_command)), check=True, env=teacher_env)
evidence_path = WORK / 'teacher/evidence.jsonl'
evidence = [json.loads(x) for x in evidence_path.read_text().splitlines()]
for item in evidence:
    print(item['video_id'], 'error=', item.get('error'), 'calls=', item['calls'],
          'keyframes=', len(item.get('sampling', {}).get('frames', [])))
run(sys.executable, STEP_SCRIPTS / 'prepare.py', '--split', WORK / 'split',
    '--evidence', evidence_path, '--output', WORK / 'arms', '--arm', ARM)
print((WORK / 'arms/build.json').read_text())
valid = [e for e in evidence if 'graph' in e]
assert valid, 'No graph passed validation'
print(json.dumps(valid[0]['graph'], ensure_ascii=False, indent=2))
# Review the first frozen train sample, including a failure if it failed (no cherry-picking).
first = evidence[0]
print('Review video:', first['video_id'], 'status:', first.get('error', 'accepted'))
if 'sampling' in first:
    from IPython.display import display, Image
    for frame in first['sampling']['frames']:
        print(frame['id'], frame['scene'], frame['time'])
        display(Image(filename=frame['path'], width=320))


## Kiểm tra dữ liệu và train

PySceneDetect ContentDetector xác định **khoảng scene**, không re-encode video.
Hecate source port: xét mọi frame (ảnh phân tích cạnh dài tối đa 160px), histogram
HSV/cạnh 2000 chiều, global k-means++ với K=min(N//2, số scene), tối thiểu 1.
Tách subshot theo các đoạn nhãn cụm liên tiếp trong từng scene; lấy frame ít thay đổi
nhất/subshot. Không thêm frame đầu/cuối. Scene duy nhất có thể chỉ cho một keyframe.
Nguồn, giấy phép và các sửa lỗi biên/uint8 được nhúng trong HECATE_NOTICE.md/HECATE_LICENSE.
Đây là port bước keyframe, không chạy toàn bộ hệ thống thumbnail/quality filter của Hecate.
Video không có cut vẫn là một scene. Số frame không còn cố định 16 cho teacher.
Decoder timestamps được kiểm tra với CFR; dữ liệu VFR cần chuẩn hóa riêng.
Vượt budget, JSON sai hoặc xung đột danh tính được ghi rõ và loại khỏi tập train.

Parse object/attribute/relation → một lượt visual verification → merge các keyframe
kề nhau trong scene → scene event → xác minh event → đối chiếu đại diện entity trên
mọi cặp scene. Motion chỉ là thay đổi nhìn thấy giữa hai keyframe, không phải trajectory.
Frame instances và attributes theo thời gian được giữ lại. Cross-clip `same_as` không
gộp mất ID cục bộ. Không có self-consistency n lần hoặc human factual verification.
Mô tả scene dùng tối đa 4 keyframe cách đều; mọi keyframe vẫn được parse/merge.
Số lần gọi tăng theo số frame/entity/scene; xem trường `calls` trước khi chạy full.

Nhánh DAR vẫn lấy 16 frame/video. Nhánh graph dùng **đúng ảnh PNG teacher đã chọn**
và timestamp dạng text; không gán fps giả cho chuỗi ảnh không đều.
Graph là target, không xuất hiện trong user prompt. Schema hợp lệ chưa đảm bảo đúng hình ảnh.

Train full SFT LLM + aligner, freeze vision, BF16, LR 1e-5, 0.5 epoch, accumulation 32.
Context tăng lên 16384. Trước SFT, script encode toàn bộ tập bằng ms-swift thực tế;
mẫu quá dài làm dừng để điều chỉnh, không train một graph bị cắt. Chi phí VRAM cao hơn pilot cũ.
Kết quả mới còn bị ảnh hưởng bởi sampling/multi-image/context; muốn quy cải thiện cho graph
cần đối chứng cùng selected frames, accepted IDs và ngân sách train.

In [ ]:
run(sys.executable, '-c', "import torch; assert torch.cuda.is_bf16_supported(), 'Full SFT baseline requires BF16-capable GPU'")
import shutil
MODEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for location in (Path('/kaggle/working'), MODEL_OUTPUT_ROOT):
    usage = shutil.disk_usage(location)
    print('Disk', location, 'free GiB:', round(usage.free / 1024**3, 2), flush=True)
for arm in ARMS:
    run('bash', STEP_SCRIPTS / 'train.sh', extra_env={
        'MAX_LENGTH': str(MAX_LENGTH), 'BASE_MODEL': str(BASE_MODEL), 'DATA_DIR': str(WORK / 'arms'),
        'OUTPUT_ROOT': str(MODEL_OUTPUT_ROOT), 'ARM': arm,
        'NUM_TRAIN_EPOCHS': str(NUM_TRAIN_EPOCHS), 'SEED': str(SEED),
        'LEARNING_RATE': str(LEARNING_RATE),
        'GRADIENT_ACCUMULATION_STEPS': str(GRADIENT_ACCUMULATION_STEPS)})


Inference một lượt, không dùng graph teacher; sampled fps được truyền cho
Qwen2.5. Lưu raw text, frame hash và token/time metadata. Không retry chọn output đẹp.

In [ ]:
predictions = []
model_checkpoints = {}
for arm in ARMS:
    checkpoints = list((MODEL_OUTPUT_ROOT / f'{arm}-seed{SEED}').rglob('config.json'))
    checkpoints = [p.parent for p in checkpoints
                   if p.parent.name.startswith('checkpoint-') and p.parent.name.split('-')[-1].isdigit()]
    assert checkpoints, f'No full checkpoint saved for {arm}'
    checkpoints = [max(checkpoints, key=lambda p: int(p.name.split('-')[-1]))]
    assert list(checkpoints[0].glob('*.safetensors')), 'Missing full checkpoint weights'
    assert not (checkpoints[0] / 'adapter_config.json').exists(), 'Expected full SFT checkpoint'
    model_checkpoints[arm] = checkpoints[0]
    output = WORK / f'{arm}-predictions.jsonl'
    run(sys.executable, SCRIPTS / 'run.py', 'predict', '--model', checkpoints[0],
        '--input', WORK / 'split/dev.jsonl', '--output', output)
    predictions.append(f'{arm}={output}')
run(sys.executable, SCRIPTS / 'evaluate.py', '--manifest', WORK / 'split/dev.jsonl',
    '--predictions', *predictions, '--reference', ARM, '--output', WORK / 'comparison.json')
print((WORK / 'comparison.json').read_text())
if MODE == 'full':
    source = model_checkpoints[ARM]
    destination = WORK / 'final_checkpoint'
    size = sum(p.stat().st_size for p in source.rglob('*') if p.is_file())
    free = shutil.disk_usage(WORK).free
    print('Final checkpoint GiB:', round(size / 1024**3, 2),
          'working free GiB:', round(free / 1024**3, 2), flush=True)
    assert free > size + 512 * 1024**2, 'Not enough persistent space for final checkpoint'
    shutil.copytree(source, destination)
    assert list(destination.glob('*.safetensors'))
    print('Persistent full checkpoint:', destination, flush=True)


## Đọc kết quả

Inference/evaluation giữ quy trình DAR của bản gốc. Một arm chưa đủ kết luận cải thiện.
So sánh trên cùng dev video và accepted train IDs; không dùng test để chọn cấu hình.
`stsg_no_links` giữ entities/attributes/scene intervals, bỏ motion/reference/event links:
đây là ablation về cạnh, không loại mọi thông tin thời gian vì timestamps vẫn còn.

Artifacts: `teacher/run.json`, từng `sampling.json`, `calls/*.json`, `intermediate.json`
(frame graphs + clip graphs), `graph.json`, `arms/build.json`, `arms/*-lengths.json`,
checkpoint và DAR dev predictions. Re-run dùng WORK/OUTPUT mới để tránh trộn kết quả.
Các cell unit test dùng video tổng hợp và fake teacher để kiểm tra wiring. Cần chạy
smoke với checkpoint thật và kiểm tra graph bằng mắt trước khi tăng quy mô.

Nguồn: [STEP §3–4](https://arxiv.org/html/2412.00161v2),
[VILA 1.5 native inference](https://github.com/NVlabs/VILA/tree/6b941da19e31ddfdfaa60160908ccf0978d96615),
[VILA3B model card](https://huggingface.co/Efficient-Large-Model/VILA1.5-3b).
Đây là graph-supervised auxiliary SFT, chưa phải graph encoder hoặc STEP QRA self-training.